# Megaminx beam search on Kaggle TPU v5e-8 - AZ v4 V-only shareable (JAX SPMD, B=48M, 720 syms)

Multi-collaborator TPU beam search for the
[CayleyPy Megaminx Kaggle competition](https://www.kaggle.com/competitions/cayley-py-megaminx).
**6x larger beam than the 8M shareable**, 1.5x larger than the 32M shareable,
**2x the symmetry group** for extra ensemble diversity. Each collaborator
runs a chunk of pids on their own TPU quota; results union-merged via
per-pid min.

## What's new vs the 48M shareable

This variant attaches the `megaminx-rotations-720` dataset, which contains
all 720 valid sticker permutations of Megaminx (60 rotations × 6 internal
relabelings + 60 reflections × 6 = 720). The original 360-entry
rotations.npy held only sign-preserving symmetries; the new 360 are
sign-flipping (reflection) symmetries, which give the V model a different
"view" of each scrambled state. The notebook's conj_idx logic accepts
sign-flipping rotations natively (no solve-side change).

Also adds the `K_SYM` + `SYM_POSITIONS` shard knobs ported from the 32M
720 variant: split a K_SYM=8 ensemble across two kernels (positions 0..3
and 4..7) and min-merge.

**Algorithm:** SPMD shared-beam at B_GLOBAL=48M, all 8 TPU v5e-8 cores
cooperating on one pid at a time. V model ranks ALL B_LOCAL * N_GEN
children per step (no qshort prefilter). 48M fits HBM via:
  * **Phase A** -- dropped per-child index arrays, donated step_fn carry.
  * **Phase B** -- packed uint32 backpointers, host np.memmap tree,
    early stop on first V0 hit.
  * **Phase C** -- streamed child generation via lax.scan over parent
    chunks (full neighbors materialization OOMs above 16M).
  * **Phase D** -- bf16 V score packed into the all-to-all bucket;
    receive side skips V re-run.
  * **One-sort receive dedup** (single argsort + sorted-order top-k).
  * **uint32 owner hash** for cheaper child-side bucket routing.

**Stack:** AZ v4 V-only (6M params, `m_az_v4_v_only.pt`).

## When to use this vs the smaller shareables

| metric | 8M shareable | 32M shareable | this (48M) |
|---|---|---|---|
| per-pid wall (steady state) | ~40 min | ~55-60 min | ~75-80 min |
| per-step (steady) | ~10-15s | ~55-60s | ~85s |
| weekly TPU quota fits | ~30 pids | ~20 pids | ~15 pids |
| beam size | 8M | 32M | 48M |
| HBM peak / rank | ~5 GB | ~13 GB | ~14 GB |
| memmap (per pid, /kaggle/working) | none | ~15 GB | ~15 GB |
| expected quality on hard pids | baseline | better | best (so far) |

Use 48M for the **hardest pids** where 32M leaves room. For broad coverage
use 32M; for raw throughput use 8M.

## Per-pid wall (TPU v5e-8, B_GLOBAL=48M, alpha=2)

| measure | value |
|---|---|
| compile (first pid) | ~60-90s |
| per-step (steady) | ~85s |
| per-beam-call wall (55-step solve) | ~78 min |
| per-beam-call wall (NUM_STEPS=80 cap) | ~115 min |

Total beam calls per pid = `K_SYM * (2 if use_niss else 1)`.

## Recommended scopes per kernel

Total beam calls = `N_PIDS × len(SYM_POSITIONS) × (2 if use_niss else 1)`.
Each call ≈ 78 min. Keep total beam calls ≤ 6 for the 9h kernel budget.

| use_niss | scope (N_PIDS × N_POSITIONS) | wall (incl. compile) |
|---|---|---|
| 0 | 4 × 1                  | ~5.5 h |
| 0 | 6 × 1                  | ~8 h |
| 0 | 2 × 4 (K=8 shard A or B) | ~5.5 h |
| 0 | 3 × 2                  | ~8 h |
| 1 | 2 × 1 (forward + inverse) | ~5.5 h |
| 1 | 3 × 1                  | ~8 h |
| 1 | 1 × 2 (K=8 shard slice with NISS) | ~5.5 h |

Better used as a targeted rescue tool on the hardest pids than full
coverage. K_SYM=8 sym-split + NISS is the most expensive setup (1 pid × 4
positions × 2 niss = 8 calls = ~10.5h, over budget — drop to 1 pid × 2
positions × 2 niss = ~5.5h).

## Sym ensemble split (K_SYM=8 across two kernels)

Separates **K_SYM** (full deterministic ensemble size, drawn from the
720-element symmetry group) from **SYM_POSITIONS** (which positions of
that ensemble THIS kernel actually runs). Both shards generate the SAME
canonical K_SYM-rotation list (seed=0) and just slice it.

K_SYM can go up to 720 in principle; common values: 4, 8, 16. The 720
set has 2x the diversity of the 360 set so larger K_SYM is more valuable
here than in the 360 variant.

K_SYM=8 across two kernels:

| kernel | K_SYM | SYM_POSITIONS | output CSV |
|---|---|---|---|
| A | 8 | `range(0, 4)` | `share_jax_submission_k8_sym0_3.csv` |
| B | 8 | `range(4, 8)` | `share_jax_submission_k8_sym4_7.csv` |

After both finish, merge via per-pid `min(path_len)` (associative):

```powershell
.venv/Scripts/python.exe megaminx/scripts/16_merge_rescue.py `
    --base megaminx/submissions/current_best.csv `
    --rescue share_jax_submission_k8_sym0_3.csv `
    --out  tmp_after_shardA.csv
.venv/Scripts/python.exe megaminx/scripts/16_merge_rescue.py `
    --base tmp_after_shardA.csv `
    --rescue share_jax_submission_k8_sym4_7.csv `
    --out  merged_k8.csv
```

## NISS -- non-inverse scramble solving

When `use_niss=1`, for each (pid, rotation) pair the V-only beam runs TWICE:
once on the rotated state `S`, and once on `invert_state(S)` (the permutation
inverse). The path found in the inverse direction is reverse-then-move-inverted
to produce a valid solution for `S`; we keep the shorter of the two paths.
Doubles wall per pair; typical heuristic gain on megaminx is 1-3% extra moves
saved on top of forward-only beam, with the wins concentrated on pids where
forward beam stalls or hits a plateau. Each direction gets its own memmap tree
file under `/kaggle/working/` (cleaned up at end of each iteration). Both
candidates verify-checked before selecting min, so a buggy inversion can
never silently corrupt the submission.

## How to run as a collaborator

1. **Fork** this notebook on Kaggle ("Copy and Edit").
2. Two datasets auto-attach (verify in the right sidebar -> Data -> Input):
   - `artgor/megaminx-tpu-artifacts` (must contain `m_az_v4_v_only.pt`).
   - `artgor/megaminx-rotations-720` (must contain `rotations_720.npy`).
3. **Edit Cell 1 (CONFIG)** to set `START_PID`, `END_PID`, `K_SYM`,
   `SYM_POSITIONS`, `use_niss`. Pair with another collaborator on a
   K_SYM=8 split: agree on the pid range, then take complementary
   `SYM_POSITIONS` slices.
4. **Switch accelerator to TPU v5e-8** in the right sidebar.
5. Run all cells.
6. Per-iteration partial saves -- even if Kaggle kills mid-run, every
   completed (pid, sym_pos) pair is in
   `/kaggle/working/share_jax_partial_<SLICE_TAG>.json`.
7. After completion, download
   `/kaggle/working/share_jax_submission_<SLICE_TAG>.csv` (and the matching
   `_final_<SLICE_TAG>.json` if convenient) and send back to the owner
   for cross-shard min-merging.

## Notes for collaborators

- The 9h Kaggle TPU kernel kill is real; the wall projection in Cell 2
  will warn if your config exceeds 8.5h.
- Weekly TPU quota is 20h per account.
- NUM_STEPS=80 caps the memmap at 15.4 GB (under /kaggle/working's 20 GB).
- /kaggle/working tree memmap files (~15 GB per active beam call)
  auto-delete in the per-iteration cleanup.
- `verify=True` in the per-pid output line is the integrity check
  (path applied to init state produces V0). Should always be True for
  found pids; report if you see verify=False.
- **Both shards of a K=8 split MUST use the same `K_SYM`** (and the same
  hardcoded `SYM_SEED=0`), otherwise their position indices refer to
  different rotations and the merge is meaningless.


In [ ]:
# ============================================================================
# CONFIG - EDIT THIS CELL, then Run All
# ============================================================================
#
# Test set has 1001 pids (0..1000), ordered by random-walk length
# (pid 0 = easiest scramble, pid 1000 = hardest).
#
# Knobs:
#   START_PID, END_PID  - pid range to solve (half-open).
#   K_SYM               - FULL target sym-ensemble size, drawn from the
#                         720-element symmetry group. Practical: 1..16.
#                         Defines the deterministic canonical K_SYM-rotation
#                         list (seed=0). Both shards of a split run MUST set
#                         the same K_SYM, otherwise their position indices
#                         refer to different rotations.
#   SYM_POSITIONS       - which positions of the K_SYM ensemble THIS kernel
#                         actually runs (a range or list of ints in
#                         [0, K_SYM)). Lets one kernel run a slice and
#                         another run the rest; min-merge their CSVs.
#                         Memory does NOT scale with len(SYM_POSITIONS).
#   use_niss            - 0 = forward beam only.
#                         1 = also solve invert_state(S) and take the
#                         shorter path. Doubles wall PER (pid, sym_pos).
#                         Combines multiplicatively: total beam calls
#                         = N_PIDS * len(SYM_POSITIONS) * (2 if use_niss else 1).
#
# Split recipes (B=48M, ~78 min/beam-call; keep total beam calls <= 6):
#
#   K=8 across two kernels (recommended for hardest pids, no NISS):
#     Kernel A:  K_SYM=8, SYM_POSITIONS=range(0, 4), END_PID-START_PID=1, use_niss=0
#     Kernel B:  K_SYM=8, SYM_POSITIONS=range(4, 8), END_PID-START_PID=1, use_niss=0
#     1 pid x 4 calls x 1 niss = 4 beam calls per kernel ≈ 5.5 h.
#
#   K=8 split + NISS on the very hardest pid (most thorough recipe):
#     Kernel A:  K_SYM=8, SYM_POSITIONS=range(0, 2), END_PID-START_PID=1, use_niss=1
#     Kernel B:  K_SYM=8, SYM_POSITIONS=range(2, 4), END_PID-START_PID=1, use_niss=1
#     Kernel C:  K_SYM=8, SYM_POSITIONS=range(4, 6), END_PID-START_PID=1, use_niss=1
#     Kernel D:  K_SYM=8, SYM_POSITIONS=range(6, 8), END_PID-START_PID=1, use_niss=1
#     1 pid x 2 positions x 2 niss = 4 beam calls per kernel ≈ 5.5 h.
#
#   K=4 in one kernel (smaller ensemble, no split):
#     K_SYM=4, SYM_POSITIONS=range(0, 4)  (1 pid ≈ 5.5h; 2 pids over budget)
#
#   Smoke / identity only:
#     K_SYM=1, SYM_POSITIONS=range(0, 1)  (up to 6 pids in 9h)
# ============================================================================

# BRIDGE-RESIDUAL DEFAULTS — residuals are SHALLOW (≤ window_len, typically ≤60)
# so we don't need 48M beams. Each residual at B=1M with sym4+NISS takes
# ~15-30 min on TPU v5e-8 (depends on actual bridge depth — beam terminates
# early when a shorter path is found).
#
# Defaults sized for 9h Kaggle TPU budget:
#   K_SYM=4 + NISS=1 = 8 sub-solves per residual × ~3 min each = ~24 min/residual
#   → ~15-18 residuals fit in 9h (allow ~1h compile + setup overhead).
# Bump END_PID for more, but watch kernel wall and use SLICE_TAG to merge
# multiple kernel runs.
START_PID = 0
END_PID = 15                  # process the first 15 residuals — safe within 9h.
                              # Bump up or split via START_PID/END_PID across kernels.
K_SYM = 4                     # sym-ensemble — see notebook README for K=8 split
SYM_POSITIONS = range(0, 4)   # all positions in one kernel for K_SYM=4
use_niss = 1                  # bridge variant: NISS doubles diversity at 2x cost

B_GLOBAL = 1 * 1024 * 1024    # 1M (vs 48M for pid kernel — residuals are shallow)
ALPHA_QSHORT = 2              # receive-side 2x oversampling for quality
INTERNAL_BS = 16384
NUM_STEPS = 80                # bridges are typically ≤60 — cap is conservative

PARENT_CHUNK = 131072         # B_LOCAL / PARENT_CHUNK = 48 chunks per step
PACK_V_SCORE = True           # bf16 score packed into bucket; skip recv V re-run
PROGRESS_EVERY = 5            # print per-step timing every N steps

# ============================================================================
# Do not edit below this line
# ============================================================================


In [ ]:
# Cell 2: Setup. JAX_ENABLE_X64 + XLA_PYTHON_CLIENT_MEM_FRACTION must be set
# BEFORE `import jax`. Phase A: raise the JAX/XLA memory fraction from the
# default 0.75 to 0.95 -- frees ~3 GB of headroom at the cost of less
# scheduler slack.
import sys, os, time, json
os.environ["JAX_ENABLE_X64"] = "True"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, x):
        for s in self.streams:
            s.write(x); s.flush()
    def flush(self):
        for s in self.streams:
            s.flush()

_LOG = open("/kaggle/working/run.log", "w", encoding="utf-8")
sys.stdout = _Tee(sys.__stdout__, _LOG)
sys.stderr = _Tee(sys.__stderr__, _LOG)

import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_enable_x64", True)
print(f"jax {jax.__version__}  x64={jax.config.jax_enable_x64}")
_t = jnp.int64(3952374295417501729)
assert _t.dtype == jnp.int64 and int(_t) == 3952374295417501729, "x64 NOT enabled"

try:
    from jax.experimental.shard_map import shard_map  # noqa: F401
except ImportError:
    from jax import shard_map  # JAX 0.5+
from jax.sharding import Mesh, PartitionSpec, NamedSharding  # noqa: F401

devices = jax.devices()
print(f"devices: {devices}")
N_DEVICES = len(devices)
assert N_DEVICES == 8, f"Expected 8 TPU cores; got {N_DEVICES}. Switch accelerator to TPU v5e-8."

assert 0 <= START_PID < END_PID <= 15, \
    f"invalid pid range: START_PID={START_PID}, END_PID={END_PID}"
assert isinstance(K_SYM, int) and 1 <= K_SYM <= 720, \
    f"K_SYM must be in [1, 720]; got {K_SYM}"
SYM_POSITIONS = list(SYM_POSITIONS)
assert len(SYM_POSITIONS) >= 1, "SYM_POSITIONS must be non-empty"
assert all(isinstance(p, int) and 0 <= p < K_SYM for p in SYM_POSITIONS), \
    f"SYM_POSITIONS={SYM_POSITIONS} contains values outside [0, K_SYM={K_SYM})"
assert len(set(SYM_POSITIONS)) == len(SYM_POSITIONS), \
    f"SYM_POSITIONS has duplicate positions: {SYM_POSITIONS}"
N_LOCAL_SYM = len(SYM_POSITIONS)
assert use_niss in (0, 1), f"use_niss must be 0 or 1; got {use_niss}"
assert B_GLOBAL % N_DEVICES == 0
B_LOCAL = B_GLOBAL // N_DEVICES
K_PER_PEER = (ALPHA_QSHORT * B_LOCAL) // N_DEVICES
assert B_LOCAL % PARENT_CHUNK == 0, \
    f"PARENT_CHUNK={PARENT_CHUNK} must divide B_LOCAL={B_LOCAL}"
N_PIDS = END_PID - START_PID
NISS_FACTOR = 2 if use_niss else 1
print(f"\n== CONFIG ==")
print(f"pid range: [{START_PID}, {END_PID})  ({N_PIDS} pids)")
print(f"K_SYM (full ensemble): {K_SYM}")
print(f"SYM_POSITIONS (this kernel): {SYM_POSITIONS}  ({N_LOCAL_SYM} positions)")
print(f"use_niss: {use_niss}  (beam calls per (pid, sym_pos) = {NISS_FACTOR})")
print(f"B_GLOBAL: {B_GLOBAL:,}  B_LOCAL: {B_LOCAL:,}  K_PER_PEER: {K_PER_PEER:,}")
print(f"NUM_STEPS: {NUM_STEPS}  alpha: {ALPHA_QSHORT}")
print(f"PARENT_CHUNK: {PARENT_CHUNK:,}  n_chunks_per_step: {B_LOCAL // PARENT_CHUNK}")

# Per-beam-call wall estimate (linear-ish in B_GLOBAL).
_per_call = {1*1024*1024: 450, 4*1024*1024: 1300, 8*1024*1024: 2600,
             16*1024*1024: 5200, 32*1024*1024: 10800,
             48*1024*1024: 15600}.get(B_GLOBAL, 15600)
_n_pairs = N_PIDS * N_LOCAL_SYM
_n_beam_calls = _n_pairs * NISS_FACTOR
_proj = 0.1 + (_n_beam_calls * _per_call) / 3600
print(f"estimated wall: {_proj:.1f}h (compile + {_n_beam_calls} beam calls @ ~{_per_call}s each)")
if _proj > 8.5:
    print(f"WARNING: estimated wall {_proj:.1f}h exceeds 8.5h target.")
    print(f"  Kaggle kills TPU kernels around 9h. Consider reducing pid range,")
    print(f"  shrinking SYM_POSITIONS, or setting use_niss=0.")


In [ ]:
# Cell 3: Puzzle + all 1001 pid states embedded.
PUZZLE_INFO = json.loads('{"central_state":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"generators":{"U":[3,4,5,18,19,20,0,1,2,9,10,11,12,13,14,15,16,17,21,22,23,6,7,8,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,62,63,74,75,64,65,66,67,68,69,70,71,76,77,72,73,60,61,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"-U":[6,7,8,0,1,2,21,22,23,9,10,11,12,13,14,15,16,17,3,4,5,18,19,20,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,76,77,60,61,64,65,66,67,68,69,70,71,74,75,62,63,72,73,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"D":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,54,55,56,36,37,38,42,43,44,45,46,47,48,49,50,39,40,41,57,58,59,51,52,53,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,114,115,92,93,96,97,98,99,100,101,102,103,104,105,106,107,108,109,94,95,112,113,116,117,110,111,118,119],"-D":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,39,40,41,51,52,53,42,43,44,45,46,47,48,49,50,57,58,59,36,37,38,54,55,56,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,94,95,110,111,96,97,98,99,100,101,102,103,104,105,106,107,108,109,116,117,112,113,92,93,114,115,118,119],"F":[8,6,7,3,4,5,31,32,30,1,2,0,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,34,35,33,10,11,9,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,65,64,62,63,87,86,78,79,68,69,70,71,72,73,74,75,76,77,61,60,80,81,82,83,84,85,67,66,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"-F":[11,9,10,3,4,5,1,2,0,35,33,34,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,8,6,7,32,30,31,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,79,78,62,63,61,60,87,86,68,69,70,71,72,73,74,75,76,77,66,67,80,81,82,83,84,85,65,64,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"B":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,40,41,39,25,26,24,30,31,32,33,34,35,36,37,38,53,51,52,42,43,44,28,29,27,48,49,50,46,47,45,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,111,110,96,97,100,101,102,103,104,105,99,98,108,109,113,112,107,106,114,115,116,117,118,119],"-B":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,29,27,28,47,45,46,30,31,32,33,34,35,36,37,38,26,24,25,42,43,44,53,51,52,48,49,50,40,41,39,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,98,99,107,106,100,101,102,103,104,105,113,112,108,109,97,96,111,110,114,115,116,117,118,119],"L":[10,11,9,2,0,1,6,7,8,13,14,12,16,17,15,4,5,3,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,78,79,64,65,66,67,80,81,62,63,72,73,74,75,76,77,68,69,70,71,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"-L":[4,5,3,17,15,16,6,7,8,2,0,1,11,9,10,14,12,13,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,70,71,64,65,66,67,78,79,80,81,72,73,74,75,76,77,62,63,68,69,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"DR":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,46,47,45,52,53,51,43,44,42,59,57,58,54,55,56,49,50,48,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,102,103,112,113,104,105,106,107,108,109,110,111,116,117,114,115,118,119,100,101],"-DR":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,50,48,49,44,42,43,59,57,58,47,45,46,54,55,56,52,53,51,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,118,119,100,101,104,105,106,107,108,109,110,111,102,103,114,115,112,113,116,117],"BL":[0,1,2,16,17,15,6,7,8,9,10,11,12,13,14,25,26,24,5,3,4,21,22,23,28,29,27,19,20,18,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,83,82,72,73,71,70,76,77,78,79,80,81,99,98,74,75,86,87,88,89,90,91,92,93,94,95,96,97,85,84,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"-BL":[0,1,2,19,20,18,6,7,8,9,10,11,12,13,14,5,3,4,29,27,28,21,22,23,17,15,16,26,24,25,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,75,74,72,73,84,85,76,77,78,79,80,81,71,70,99,98,86,87,88,89,90,91,92,93,94,95,96,97,83,82,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"FR":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,49,50,48,31,32,30,36,37,38,39,40,41,42,43,44,45,46,47,58,59,57,51,52,53,34,35,33,56,54,55,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,108,109,87,86,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,119,118,110,111,112,113,89,88,116,117,115,114],"-FR":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,35,33,34,56,54,55,36,37,38,39,40,41,42,43,44,45,46,47,32,30,31,51,52,53,58,59,57,50,48,49,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,89,88,115,114,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,86,87,110,111,112,113,119,118,116,117,109,108],"BR":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,28,29,27,20,18,19,24,25,26,46,47,45,30,31,32,33,34,35,36,37,38,39,40,41,22,23,21,43,44,42,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,85,84,74,75,76,77,78,79,80,81,82,83,107,106,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,105,104,73,72,102,103,108,109,110,111,112,113,114,115,116,117,118,119],"-BR":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,22,23,21,44,42,43,24,25,26,20,18,19,30,31,32,33,34,35,36,37,38,39,40,41,47,45,46,29,27,28,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,105,104,74,75,76,77,78,79,80,81,82,83,73,72,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,106,107,103,102,85,84,108,109,110,111,112,113,114,115,116,117,118,119],"FL":[0,1,2,3,4,5,6,7,8,34,35,33,10,11,9,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,55,56,54,13,14,12,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,38,36,37,57,58,59,60,61,62,63,64,65,89,88,67,66,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,92,93,69,68,91,90,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"-FL":[0,1,2,3,4,5,6,7,8,14,12,13,38,36,37,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,11,9,10,55,56,54,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,35,33,34,57,58,59,60,61,62,63,64,65,69,68,91,90,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,67,66,93,92,88,89,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"R":[0,1,2,3,4,5,23,21,22,9,10,11,12,13,14,15,16,17,18,19,20,43,44,42,24,25,26,27,28,29,7,8,6,33,34,35,36,37,38,39,40,41,49,50,48,45,46,47,31,32,30,51,52,53,54,55,56,57,58,59,60,61,62,63,76,77,66,67,68,69,70,71,72,73,74,75,104,105,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,108,109,102,103,100,101,106,107,64,65,110,111,112,113,114,115,116,117,118,119],"-R":[0,1,2,3,4,5,32,30,31,9,10,11,12,13,14,15,16,17,18,19,20,7,8,6,24,25,26,27,28,29,50,48,49,33,34,35,36,37,38,39,40,41,23,21,22,45,46,47,44,42,43,51,52,53,54,55,56,57,58,59,60,61,62,63,108,109,66,67,68,69,70,71,72,73,74,75,64,65,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,104,105,102,103,76,77,106,107,100,101,110,111,112,113,114,115,116,117,118,119],"DL":[0,1,2,3,4,5,6,7,8,9,10,11,37,38,36,13,14,12,18,19,20,21,22,23,16,17,15,27,28,29,30,31,32,33,34,35,41,39,40,25,26,24,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,90,91,80,81,84,85,86,87,88,89,94,95,92,93,96,97,82,83,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"-DL":[0,1,2,3,4,5,6,7,8,9,10,11,17,15,16,26,24,25,18,19,20,21,22,23,41,39,40,27,28,29,30,31,32,33,34,35,14,12,13,37,38,36,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,82,83,96,97,84,85,86,87,88,89,80,81,92,93,90,91,94,95,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119]}}')
GENERATORS = PUZZLE_INFO["generators"]
SOLVED = tuple(PUZZLE_INFO["central_state"])
MOVE_NAMES = list(GENERATORS.keys())
N_GEN = len(MOVE_NAMES)
STATE_SIZE = len(SOLVED)
print(f"N_GEN={N_GEN}  STATE_SIZE={STATE_SIZE}")

all_moves_np = np.array([GENERATORS[n] for n in MOVE_NAMES], dtype=np.int32)
all_moves = jnp.asarray(all_moves_np)
V0_np = np.array(SOLVED, dtype=np.int8)
V0 = jnp.asarray(V0_np)

# NISS support: per-move-index inverse. Megaminx convention: "name" and "-name"
# are inverse generators. INV_MOVE_IDX[i] = index of inverse of MOVE_NAMES[i].
INV_MOVE_IDX = np.full(N_GEN, -1, dtype=np.int32)
_move_name_to_idx = {n: i for i, n in enumerate(MOVE_NAMES)}
for _i, _nm in enumerate(MOVE_NAMES):
    _inv = _nm[1:] if _nm.startswith("-") else "-" + _nm
    if _inv not in _move_name_to_idx:
        raise ValueError(f"missing inverse generator for {_nm} (expected {_inv})")
    INV_MOVE_IDX[_i] = _move_name_to_idx[_inv]
assert (INV_MOVE_IDX >= 0).all() and INV_MOVE_IDX[INV_MOVE_IDX].tolist() == list(range(N_GEN))
print(f"INV_MOVE_IDX built: {N_GEN} move pairs verified involutive")

_ALL_PID_STATES = json.loads('{"0":[6,7,8,0,1,2,9,10,11,35,33,34,12,13,14,46,47,45,26,24,25,19,20,18,29,27,28,53,51,52,50,48,49,32,30,31,36,37,38,39,40,41,23,21,22,4,5,3,44,42,43,17,15,16,54,55,56,57,58,59,76,77,60,61,108,109,87,86,68,69,70,71,85,84,78,79,63,62,66,67,80,81,83,82,103,102,65,64,88,89,90,91,92,93,94,95,96,97,98,99,104,105,73,72,74,75,112,113,100,101,110,111,107,106,114,115,116,117,118,119],"1":[18,19,20,3,4,5,30,31,32,9,10,11,12,13,14,15,16,17,0,1,2,49,50,48,40,41,39,25,26,24,59,57,58,33,34,35,36,37,38,27,28,29,8,6,7,44,42,43,21,22,23,51,52,53,54,55,56,45,46,47,74,75,62,63,65,64,66,67,68,69,70,71,105,104,72,73,61,60,78,79,80,81,82,83,118,119,86,87,88,89,90,91,92,93,94,95,111,110,96,97,103,102,77,76,113,112,99,98,109,108,107,106,84,85,114,115,116,117,100,101],"2":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,21,22,23,27,28,29,47,45,46,51,52,53,35,33,34,38,36,37,59,57,58,26,24,25,31,32,30,20,18,19,49,50,48,42,43,44,39,40,41,55,56,54,60,61,62,63,64,65,66,67,68,69,70,71,72,73,85,84,76,77,78,79,80,81,82,83,105,104,89,88,93,92,90,91,110,111,113,112,98,99,107,106,109,108,74,75,102,103,117,116,86,87,97,96,101,100,94,95,119,118,115,114],"3":[3,4,5,18,19,20,0,1,2,9,10,11,12,13,14,15,16,17,6,7,8,27,28,29,47,45,46,51,52,53,35,33,34,38,36,37,59,57,58,26,24,25,21,22,23,43,44,42,32,30,31,54,55,56,39,40,41,50,48,49,62,63,74,75,64,65,66,67,68,69,70,71,76,77,85,84,60,61,78,79,80,81,82,83,105,104,89,88,93,92,90,91,110,111,113,112,98,99,107,106,72,73,101,100,102,103,117,116,86,87,97,96,119,118,94,95,115,114,109,108],"4":[17,15,16,28,29,27,4,5,3,10,11,9,30,31,32,14,12,13,20,18,19,6,7,8,24,25,26,35,33,34,45,46,47,49,50,48,36,37,38,53,51,52,2,0,1,21,22,23,44,42,43,39,40,41,54,55,56,57,58,59,70,71,74,75,64,65,89,88,66,67,80,81,79,78,85,84,61,60,62,63,68,69,82,83,107,106,109,108,87,86,90,91,92,93,94,95,96,97,98,99,104,105,77,76,73,72,102,103,100,101,117,116,112,113,114,115,110,111,118,119],"5":[33,34,35,9,10,11,30,31,32,5,3,4,19,20,18,16,17,15,6,7,8,21,22,23,41,39,40,27,28,29,26,24,25,1,2,0,14,12,13,37,38,36,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,66,67,76,77,99,98,87,86,85,84,78,79,72,73,68,69,65,64,81,80,71,70,96,97,75,74,62,63,88,89,60,61,92,93,90,91,94,95,83,82,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119],"6":[33,34,35,9,10,11,7,8,6,5,3,4,19,20,18,16,17,15,28,29,27,22,23,21,41,39,40,46,47,45,26,24,25,1,2,0,14,12,13,37,38,36,44,42,43,50,48,49,31,32,30,51,52,53,54,55,56,57,58,59,66,67,104,105,99,98,87,86,85,84,78,79,74,75,68,69,77,76,81,80,71,70,96,97,107,106,62,63,88,89,60,61,92,93,90,91,94,95,83,82,108,109,101,100,73,72,102,103,64,65,110,111,112,113,114,115,116,117,118,119],"7":[6,7,8,0,1,2,9,10,11,35,33,34,12,13,14,51,52,53,28,29,27,19,20,18,46,47,45,39,40,41,50,48,49,32,30,31,36,37,38,26,24,25,5,3,4,15,16,17,21,22,23,59,57,58,54,55,56,42,43,44,76,77,60,61,108,109,87,86,68,69,70,71,85,84,78,79,63,62,66,67,80,81,83,82,103,102,65,64,88,89,90,91,92,93,94,95,98,99,107,106,73,72,112,113,74,75,111,110,100,101,97,96,116,117,114,115,118,119,104,105],"8":[5,3,4,16,17,15,30,31,32,9,10,11,12,13,14,25,26,24,0,1,2,49,50,48,28,29,27,43,44,42,59,57,58,33,34,35,36,37,38,39,40,41,8,6,7,53,51,52,21,22,23,20,18,19,54,55,56,45,46,47,71,70,62,63,65,64,66,67,68,69,83,82,105,104,72,73,61,60,78,79,80,81,99,98,118,119,86,87,88,89,90,91,92,93,94,95,96,97,85,84,103,102,77,76,113,112,75,74,109,108,110,111,106,107,114,115,116,117,100,101],"9":[34,35,33,50,48,49,59,57,58,32,30,31,10,11,9,12,13,14,18,19,20,21,22,23,16,17,15,27,28,29,2,0,1,3,4,5,38,36,37,25,26,24,46,47,45,52,53,51,43,44,42,55,56,54,40,41,39,8,6,7,64,65,118,119,86,87,67,66,90,91,71,70,72,73,74,75,63,62,77,76,79,78,80,81,84,85,108,109,92,93,69,68,95,94,96,97,82,83,98,99,102,103,112,113,104,105,106,107,61,60,110,111,116,117,89,88,115,114,100,101],"10":[11,9,10,50,48,49,59,57,58,13,14,12,38,36,37,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,35,33,34,31,32,30,1,2,0,39,40,41,46,47,45,52,53,51,43,44,42,55,56,54,4,5,3,6,7,8,64,65,118,119,67,66,69,68,63,62,71,70,72,73,74,75,60,61,77,76,80,81,82,83,84,85,108,109,78,79,93,92,88,89,94,95,96,97,98,99,102,103,112,113,104,105,106,107,86,87,110,111,116,117,91,90,115,114,100,101],"11":[34,35,33,3,4,5,6,7,8,1,2,0,14,12,13,53,51,52,18,19,20,21,22,23,41,39,40,27,28,29,17,15,16,26,24,25,57,58,59,54,55,56,42,43,44,45,46,47,32,30,31,38,36,37,9,10,11,50,48,49,69,68,88,89,64,65,63,62,81,80,70,71,72,73,74,75,76,77,61,60,82,83,96,97,84,85,90,91,79,78,66,67,93,92,114,115,94,95,98,99,100,101,102,103,104,105,106,107,86,87,111,110,112,113,119,118,117,116,109,108],"12":[48,49,50,15,16,17,34,35,33,1,2,0,12,13,14,4,5,3,18,19,20,21,22,23,24,25,26,46,47,45,7,8,6,10,11,9,52,53,51,36,37,38,43,44,42,30,31,32,28,29,27,39,40,41,54,55,56,57,58,59,60,61,109,108,87,86,78,79,68,69,62,63,72,73,74,75,64,65,70,71,80,81,82,83,84,85,67,66,88,89,90,91,117,116,92,93,96,97,98,99,118,119,76,77,104,105,102,103,100,101,94,95,112,113,114,115,110,111,106,107],"13":[48,49,50,15,16,17,34,35,33,1,2,0,12,13,14,4,5,3,28,29,27,20,18,19,24,25,26,44,42,43,7,8,6,10,11,9,36,37,38,39,40,41,23,21,22,30,31,32,47,45,46,57,58,59,52,53,51,54,55,56,60,61,109,108,87,86,78,79,68,69,62,63,85,84,74,75,64,65,70,71,80,81,82,83,107,106,67,66,88,89,90,91,92,93,94,95,96,97,98,99,118,119,76,77,73,72,105,104,100,101,110,111,112,113,117,116,114,115,102,103],"14":[48,49,50,0,1,2,59,57,58,9,10,11,12,13,14,15,16,17,47,45,46,32,30,31,28,29,27,21,22,23,42,43,44,8,6,7,54,55,56,41,39,40,38,36,37,3,4,5,20,18,19,26,24,25,34,35,33,51,52,53,101,100,60,61,76,77,66,67,68,69,83,82,85,84,62,63,119,118,78,79,80,81,75,74,104,105,64,65,87,86,90,91,114,115,92,93,111,110,98,99,116,117,70,71,109,108,72,73,102,103,94,95,97,96,89,88,107,106,113,112]}')
ALL_PID_STATES = {int(k): v for k, v in _ALL_PID_STATES.items()}
TEST_PIDS = list(range(START_PID, END_PID))
PID_STATES = {pid: ALL_PID_STATES[pid] for pid in TEST_PIDS}
print(f"selected {len(TEST_PIDS)} pids: [{TEST_PIDS[0]}..{TEST_PIDS[-1]}]")


In [ ]:
# Cell 4: Load m_az_v4_v_only.pt and rotations_720.npy from Kaggle datasets.
import torch
from pathlib import Path

# Dataset 1: model checkpoint (artgor/megaminx-tpu-artifacts).
dataset_root = None
for cand in [
    Path("/kaggle/input/megaminx-tpu-artifacts"),
    Path("/kaggle/input/datasets/artgor/megaminx-tpu-artifacts"),
]:
    if cand.exists():
        dataset_root = cand
        break
if dataset_root is None:
    raise SystemExit("megaminx-tpu-artifacts dataset not found in /kaggle/input. "
                     "If you forked this notebook, the dataset should auto-attach. "
                     "Check the right sidebar -> Data -> Input.")
print("model dataset:", dataset_root)

v_path = dataset_root / "m_az_v4_v_only.pt"
if not v_path.exists():
    raise SystemExit(
        f"missing artifact: {v_path}\n"
        f"The dataset must include m_az_v4_v_only.pt (added 2026-05-11 with the\n"
        f"AZ v4 V-only breakthrough). If you forked an older notebook copy, re-attach\n"
        f"the latest version of artgor/megaminx-tpu-artifacts via the right sidebar."
    )
print(f"V model: {v_path}")

# Dataset 2: 720-rotation sym set (artgor/megaminx-rotations-720).
rotations_dataset_root = None
for cand in [
    Path("/kaggle/input/megaminx-rotations-720"),
    Path("/kaggle/input/datasets/artgor/megaminx-rotations-720"),
]:
    if cand.exists():
        rotations_dataset_root = cand
        break
if rotations_dataset_root is None:
    raise SystemExit("megaminx-rotations-720 dataset not found in /kaggle/input. "
                     "Attach artgor/megaminx-rotations-720 in the right sidebar -> Data.")
print("rotations dataset:", rotations_dataset_root)


In [ ]:
# Cell 5: Pure-JAX inference (inlined from jax_model.py).

"""Pure-JAX inference for ResMLPDistance.

The model used in the megaminx solver stack is:
  Embedding(num_classes=120, embed_dim=16)
  -> flatten -> Linear(in=1920, hidden_dims[0]) -> LayerNorm -> ReLU
  -> Linear(hidden_dims[0], hidden_dims[1]) -> LayerNorm -> ReLU
  -> ResBlock x num_res_blocks (Linear->LN->ReLU->Linear->LN->residual->ReLU)
  -> Linear(hidden_dims[-1], output_dim)

Three variants the beam search uses:
  m_curr_v3 (V teacher):       hidden=(2048, 512),  output_dim=1   ~6M params
  m23_v2    (Q shortlister):   hidden=(2048, 1024), output_dim=24  ~12M params
  m_pi_v2   (policy):          hidden=(2048, 512),  output_dim=24  ~6M params

Functions:
  load_params_from_pt(path, hidden_dims, num_res_blocks=2)
  apply(params, x, dtype=jnp.float32)
"""
from __future__ import annotations

from pathlib import Path
from typing import Any

import jax
import jax.numpy as jnp


def _strip_orig_mod(sd: dict) -> dict:
    """Strip torch.compile's '_orig_mod.' key prefix if present."""
    return {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v
            for k, v in sd.items()}


def load_params_from_pt(
    pt_path: str | Path,
    hidden_dims: tuple[int, ...],
    num_res_blocks: int = 2,
) -> dict[str, Any]:
    """Load a PyTorch ResMLPDistance .pt and convert to a JAX params dict.

    Linear weights are transposed (PyTorch (out, in) -> JAX (in, out) so that
    `h @ w` is the canonical matmul). All params are returned as jax.numpy
    arrays in float32. Cast at the call site (apply(..., dtype=jnp.bfloat16)).
    """
    import torch
    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)
    sd = ckpt.get("state_dict", ckpt)
    sd = _strip_orig_mod(sd)

    def t(name: str) -> jnp.ndarray:
        return jnp.asarray(sd[name].float().numpy(), dtype=jnp.float32)

    params: dict[str, Any] = {
        "embed": t("embedding.weight"),  # (num_classes, embed_dim)
        "input_stack": [],
        "res_blocks": [],
        "head_w": jnp.transpose(t("head.weight"), (1, 0)),
        "head_b": t("head.bias"),
    }
    # input_stack layout: Linear, LN, ReLU per hidden_dim. Each "step" is at
    # indices [3*level, 3*level+1, (3*level+2 is ReLU, no params)].
    for level in range(len(hidden_dims)):
        lin_idx = 3 * level
        ln_idx = 3 * level + 1
        params["input_stack"].append({
            "lin_w": jnp.transpose(t(f"input_stack.{lin_idx}.weight"), (1, 0)),
            "lin_b": t(f"input_stack.{lin_idx}.bias"),
            "ln_gamma": t(f"input_stack.{ln_idx}.weight"),
            "ln_beta": t(f"input_stack.{ln_idx}.bias"),
        })
    for rb in range(num_res_blocks):
        params["res_blocks"].append({
            "lin1_w": jnp.transpose(t(f"res_blocks.{rb}.lin1.weight"), (1, 0)),
            "lin1_b": t(f"res_blocks.{rb}.lin1.bias"),
            "ln1_gamma": t(f"res_blocks.{rb}.ln1.weight"),
            "ln1_beta": t(f"res_blocks.{rb}.ln1.bias"),
            "lin2_w": jnp.transpose(t(f"res_blocks.{rb}.lin2.weight"), (1, 0)),
            "lin2_b": t(f"res_blocks.{rb}.lin2.bias"),
            "ln2_gamma": t(f"res_blocks.{rb}.ln2.weight"),
            "ln2_beta": t(f"res_blocks.{rb}.ln2.bias"),
        })
    return params


def _layer_norm(x: jnp.ndarray, gamma: jnp.ndarray, beta: jnp.ndarray,
                eps: float = 1e-5) -> jnp.ndarray:
    """PyTorch nn.LayerNorm semantics over the last dim."""
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.mean(jnp.square(x - mean), axis=-1, keepdims=True)
    x_norm = (x - mean) * jax.lax.rsqrt(var + eps)
    return x_norm * gamma + beta


def model_apply(params: dict[str, Any], x: jnp.ndarray,
          dtype: jnp.dtype = jnp.float32) -> jnp.ndarray:
    """Forward pass.

    x: (B, state_size) integer state.
    dtype: compute dtype. Embedding/output are cast to this.

    Returns (B,) for output_dim=1 (V model) or (B, output_dim) otherwise.
    """
    # Embedding lookup (params['embed'] is (num_classes, embed_dim) float32).
    embed = params["embed"]
    e = embed[x.astype(jnp.int32)]  # (B, state_size, embed_dim) float32
    h = e.reshape(e.shape[0], -1).astype(dtype)  # (B, state_size * embed_dim)

    for layer in params["input_stack"]:
        w = layer["lin_w"].astype(dtype)
        b = layer["lin_b"].astype(dtype)
        gamma = layer["ln_gamma"].astype(dtype)
        beta = layer["ln_beta"].astype(dtype)
        h = h @ w + b
        h = _layer_norm(h, gamma, beta)
        h = jax.nn.relu(h)

    for rb in params["res_blocks"]:
        skip = h
        h1 = h @ rb["lin1_w"].astype(dtype) + rb["lin1_b"].astype(dtype)
        h1 = _layer_norm(h1, rb["ln1_gamma"].astype(dtype), rb["ln1_beta"].astype(dtype))
        h1 = jax.nn.relu(h1)
        h2 = h1 @ rb["lin2_w"].astype(dtype) + rb["lin2_b"].astype(dtype)
        h2 = _layer_norm(h2, rb["ln2_gamma"].astype(dtype), rb["ln2_beta"].astype(dtype))
        h = jax.nn.relu(skip + h2)

    out = h @ params["head_w"].astype(dtype) + params["head_b"].astype(dtype)
    if out.shape[-1] == 1:
        out = jnp.squeeze(out, axis=-1)
    return out


def num_params(params: dict[str, Any]) -> int:
    """Count parameters (for sanity)."""
    n = int(params["embed"].size)
    for layer in params["input_stack"]:
        n += sum(int(v.size) for v in layer.values())
    for rb in params["res_blocks"]:
        n += sum(int(v.size) for v in rb.values())
    n += int(params["head_w"].size) + int(params["head_b"].size)
    return n


In [ ]:
# Cell 6: Single-device JAX helpers (inlined from jax_beam.py — used for the
# step-0 V eval on the 24 first-move children, which is too small to chunk).

"""Single-device JAX beam search for megaminx.

Direct port of `beam_solve_xla_qshort` (megaminx/kaggle_notebooks/tpu_beam_smoke/
build_notebook.py CELL_5_BEAM) to pure JAX. Static-shape throughout so it JITs
cleanly. Used standalone (one TPU core or CPU) and also as the foundation for the
SPMD shared-beam version in jax_beam_spmd.py.

Algorithm (qshort + policy + V0-on-hash-equality):
  Step 0: V-only on the 24 first-move children; topk -> B initial states.
          Pad to B by repeating the last chosen state.
  Steps 1..NUM_STEPS-1:
    1. Student forward (Q model, B states -> B x N_GEN logits).
    2. Policy forward (pi model, B states -> B x N_GEN logits).
    3. Generate B*N_GEN children. Hash each child. Sort+restore dedup.
    4. student_score = student_q_flat + lambda_policy * neg_log_pi_flat.
       Mask dups with +inf. topk(alpha*B, smallest) -> shortlist indices.
    5. Shortlist states. Teacher forward on shortlist -> V values.
    6. topk(B, smallest) by teacher V -> chosen B states for next step.
    7. V0 detection on chosen states via tensor equality. Record found_step
       at first hit.

Path reconstruction: walk the tree backwards from (found_step, found_pos).
"""
from __future__ import annotations

from functools import partial
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np



def make_hash_vec(state_size: int, seed: int = 0) -> np.ndarray:
    """Random int64 vector for state hashing. Returns numpy (caller moves to device)."""
    rng = np.random.default_rng(seed)
    # int64 positive values up to 1e15 — matches the torch_xla version's range.
    return rng.integers(0, int(1e15), size=state_size, dtype=np.int64)


def _model_predict_chunked(params: dict[str, Any], states: jnp.ndarray,
                           chunk_size: int, dtype: jnp.dtype) -> jnp.ndarray:
    """Chunked forward, used when (B, state_size) is too large for one pass.

    Requires states.shape[0] % chunk_size == 0 (caller ensures).
    Outputs (n,) for V or (n, num_out) for Q/policy.
    """
    n, S = states.shape
    n_chunks = n // chunk_size
    chunks = states.reshape(n_chunks, chunk_size, S)

    def scan_fn(_, chunk):
        return _, model_apply(params, chunk, dtype=dtype)

    _, outs = jax.lax.scan(scan_fn, None, chunks)
    if outs.ndim == 2:
        return outs.reshape(n)
    return outs.reshape(n, -1)


def _topk_smallest(values: jnp.ndarray, k: int):
    """Return (smallest_values, smallest_indices). jax.lax.top_k gives largest."""
    neg_v, idx = jax.lax.top_k(-values, k)
    return -neg_v, idx


@partial(jax.jit, static_argnames=(
    "B", "n_gen", "state_size", "alpha", "num_steps", "internal_bs",
    "dtype", "use_chunked_v", "use_chunked_q",
))
def beam_solve_qshort_jit(
    init_state: jnp.ndarray,         # (state_size,) int8
    teacher_params: dict[str, Any],
    student_params: dict[str, Any],
    policy_params: dict[str, Any],
    all_moves: jnp.ndarray,           # (n_gen, state_size) int32
    V0: jnp.ndarray,                  # (state_size,) int8
    hash_vec: jnp.ndarray,            # (state_size,) int64
    lambda_policy: float,
    B: int,
    alpha: int,
    n_gen: int,
    state_size: int,
    num_steps: int,
    internal_bs: int,
    dtype: Any = jnp.bfloat16,
    use_chunked_v: bool = False,
    use_chunked_q: bool = False,
):
    """Run beam search. Returns (states_final, tree_idx, tree_move, min_v_log,
    found_step, found_pos) where path is reconstructed by the caller from
    tree_idx / tree_move."""
    aB = alpha * B

    def forward_v(params, x):
        if use_chunked_v:
            return _model_predict_chunked(params, x, internal_bs, dtype)
        return model_apply(params, x, dtype=dtype)

    def forward_q(params, x):
        if use_chunked_q:
            return _model_predict_chunked(params, x, internal_bs, dtype)
        return model_apply(params, x, dtype=dtype)

    BIG_F32 = jnp.float32(1e9)

    # --- Step 0: V-only on 24 first-move children ---
    states_seed = jnp.expand_dims(init_state, 0)                       # (1, S)
    neighbors0 = states_seed[:, all_moves].reshape(-1, state_size)     # (24, S)
    # NEVER chunked at step 0: n=24 < any reasonable internal_bs, so a chunked
    # forward would reshape to (0, internal_bs, ...). Use model_apply directly.
    values0 = model_apply(teacher_params, neighbors0, dtype=dtype).astype(jnp.float32)  # (24,)
    k0 = min(B, n_gen)
    top_v0, top_idx0 = _topk_smallest(values0, k0)
    chosen0 = neighbors0[top_idx0]                                     # (k0, S)
    if k0 < B:
        pad = jnp.broadcast_to(chosen0[-1:], (B - k0, state_size))
        states = jnp.concatenate([chosen0, pad], axis=0)
    else:
        states = chosen0[:B]

    tree_idx = jnp.zeros((num_steps, B), dtype=jnp.int32)
    tree_move = jnp.full((num_steps, B), -1, dtype=jnp.int8)
    min_v_log = jnp.full((num_steps,), 1e6, dtype=jnp.float32)

    move0_full = jnp.full((B,), -1, dtype=jnp.int8)
    move0_full = move0_full.at[:k0].set(top_idx0.astype(jnp.int8))
    tree_move = tree_move.at[0].set(move0_full)

    # V0 detection at step 0 (init_state already excluded by caller).
    eq0 = jnp.all(states == V0, axis=1)
    any0 = jnp.any(eq0)
    pos0 = jnp.argmax(eq0.astype(jnp.int32))
    found_step = jnp.where(any0, jnp.int32(0), jnp.int32(-1))
    found_pos = jnp.where(any0, pos0.astype(jnp.int32), jnp.int32(-1))

    # --- Steps 1..num_steps-1 via lax.scan ---
    def beam_step(carry, j):
        states, tree_idx, tree_move, min_v_log, found_step, found_pos = carry

        # Forward student + policy on current B states.
        student_q = forward_q(student_params, states).astype(jnp.float32)   # (B, n_gen)
        policy_q = forward_q(policy_params, states).astype(jnp.float32)     # (B, n_gen)
        log_pi = policy_q - jax.scipy.special.logsumexp(policy_q, axis=-1, keepdims=True)
        neg_log_pi_flat = -log_pi.reshape(-1)                                # (B*n_gen,)

        # Generate children + hash.
        neighbors_2d = states[:, all_moves]                                  # (B, n_gen, S)
        neighbors = neighbors_2d.reshape(-1, state_size)                     # (B*n_gen, S)
        h = jnp.sum(neighbors.astype(jnp.int64) * hash_vec, axis=1)          # (B*n_gen,) int64

        # Static-shape dedup (sort+adjacent-equal+restore).
        sort_h = jnp.sort(h)
        sort_idx = jnp.argsort(h)
        is_dup_sorted = jnp.concatenate([
            jnp.zeros(1, dtype=jnp.bool_),
            sort_h[1:] == sort_h[:-1],
        ])
        restore = jnp.argsort(sort_idx)
        dup_mask = is_dup_sorted[restore]                                    # (B*n_gen,) bool

        # Student score = student_q + lambda * neg_log_pi. Mask dups.
        student_q_flat = student_q.reshape(-1)
        student_score = student_q_flat + lambda_policy * neg_log_pi_flat
        student_score_masked = jnp.where(dup_mask, BIG_F32, student_score)

        # Shortlist (alpha*B smallest by student score).
        _, shortlist_indices = _topk_smallest(student_score_masked, aB)      # (aB,) int32

        # Materialize shortlist states, run teacher rerank.
        shortlist_states = neighbors[shortlist_indices]                      # (aB, S)
        teacher_v = forward_v(teacher_params, shortlist_states).astype(jnp.float32)  # (aB,)

        # Top B by teacher V.
        top_v, chosen_local = _topk_smallest(teacher_v, B)                   # (B,)
        chosen_states = shortlist_states[chosen_local]                       # (B, S)
        chosen_global = shortlist_indices[chosen_local]                      # (B,) int32

        parent = (chosen_global // n_gen).astype(jnp.int32)
        move = (chosen_global % n_gen).astype(jnp.int8)

        # V0 detection via TENSOR equality (not hash; avoids collision risk).
        eq_v0 = jnp.all(chosen_states == V0, axis=1)
        any_hit = jnp.any(eq_v0)
        pos_hit = jnp.argmax(eq_v0.astype(jnp.int32)).astype(jnp.int32)
        is_first_hit = (found_step == -1) & any_hit
        new_found_step = jnp.where(is_first_hit, j, found_step)
        new_found_pos = jnp.where(is_first_hit, pos_hit, found_pos)

        new_tree_idx = tree_idx.at[j].set(parent)
        new_tree_move = tree_move.at[j].set(move)
        new_min_v_log = min_v_log.at[j].set(top_v[0])

        return (chosen_states, new_tree_idx, new_tree_move, new_min_v_log,
                new_found_step, new_found_pos), ()

    init_carry = (states, tree_idx, tree_move, min_v_log, found_step, found_pos)
    final_carry, _ = jax.lax.scan(beam_step, init_carry,
                                  jnp.arange(1, num_steps, dtype=jnp.int32))
    states_final, tree_idx, tree_move, min_v_log, found_step, found_pos = final_carry
    return states_final, tree_idx, tree_move, min_v_log, found_step, found_pos


def reconstruct_path(tree_idx_h: np.ndarray, tree_move_h: np.ndarray,
                     found_step: int, found_pos: int) -> list[int]:
    """Walk the tree backwards from (found_step, found_pos) to assemble the move
    sequence. Returns a list of generator indices. Strips leading -1 sentinels."""
    if found_step < 0:
        return []
    path = []
    pos = int(found_pos)
    for j in range(found_step, -1, -1):
        m = int(tree_move_h[j, pos])
        path.append(m)
        pos = int(tree_idx_h[j, pos])
    path.reverse()
    while path and path[0] < 0:
        path.pop(0)
    return path


def beam_solve_qshort(
    init_state_list: list[int],
    teacher_params: dict[str, Any],
    student_params: dict[str, Any],
    policy_params: dict[str, Any],
    all_moves: jnp.ndarray,
    V0: jnp.ndarray,
    hash_vec: jnp.ndarray,
    lambda_policy: float = 0.05,
    B: int = 131072,
    alpha: int = 2,
    n_gen: int = 24,
    state_size: int = 120,
    num_steps: int = 120,
    internal_bs: int = 32768,
    dtype: Any = jnp.bfloat16,
    use_chunked_v: bool = False,
    use_chunked_q: bool = False,
) -> dict[str, Any]:
    """High-level wrapper: prepare inputs, run jit, reconstruct path."""
    init_state = jnp.asarray(init_state_list, dtype=jnp.int8)
    if jnp.array_equal(init_state, V0):
        return {"found": True, "path_len": 0, "path_idx": [], "found_step": -1}

    (states_final, tree_idx, tree_move, min_v_log, found_step, found_pos
     ) = beam_solve_qshort_jit(
        init_state, teacher_params, student_params, policy_params,
        all_moves, V0, hash_vec, jnp.float32(lambda_policy),
        B=int(B), alpha=int(alpha), n_gen=int(n_gen), state_size=int(state_size),
        num_steps=int(num_steps), internal_bs=int(internal_bs),
        dtype=dtype,
        use_chunked_v=bool(use_chunked_v), use_chunked_q=bool(use_chunked_q),
    )
    fs = int(found_step)
    fp = int(found_pos)
    tree_idx_h = np.asarray(tree_idx)
    tree_move_h = np.asarray(tree_move)
    min_v_h = np.asarray(min_v_log).tolist()
    path = reconstruct_path(tree_idx_h, tree_move_h, fs, fp)
    return {
        "found": fs >= 0,
        "found_step": fs,
        "found_pos": fp,
        "path_idx": path,
        "path_len": len(path),
        "min_v_trajectory": min_v_h,
    }


In [ ]:
# Cell 7: SPMD V-only shared-beam (inlined from jax_beam_spmd_v_only.py).
# Same all_to_all + host-side cross-rank reduce as the qshort body, but V is
# computed on ALL children (no qshort prefilter).

"""SPMD shared-beam JAX implementation: V-ONLY variant (no qshort, no policy).

Sibling of jax_beam_spmd.py for AZ-style V models (e.g. m_az_v4_v_only) where
m23_v2 qshort REGRESSES paths because m23_v2 was distilled from m05's V
landscape, not AZ's. Drop qshort entirely; rank all B_LOCAL*N_GEN children
by the V model directly.

Algorithm (per step inside the shard_map body):
  1. Each rank generates B_LOCAL * N_GEN children of its owned states.
  2. Run V on ALL children (chunked via lax.scan).
  3. Hash each child, owner = hash % world_size.
  4. Per-owner top-K_PER_PEER by V score; pack (state, parent_local, move)
     uint8 records.
  5. Single packed `lax.all_to_all` routes bucket S from each sender to rank S.
  6. Receiver: in-rank dedup (sort+adjacent-equal+restore), RE-RUN V on
     received states (cheap vs. step 2 — aB_LOCAL ≈ 2*B_LOCAL vs. 24*B_LOCAL),
     topk(B_LOCAL) by V.
  7. V0 detection: hash equality + non-padding filter (per-rank).

Cross-rank winner selection is HOST-SIDE (cross-rank reduce inside the body
mis-propagates on TPU v5e-8; see spmd_jax_recovery.md). Each rank carries
its own local first-hit state; wrapper picks the global winner on host
after the loop.

Re-running V on receive instead of packing V scores into the bucket keeps
PACK_SIZE=128 and the packing identical to jax_beam_spmd.py — simpler code,
lower bug surface. The ~8% extra compute (aB_LOCAL vs. 24*B_LOCAL) is
negligible.
"""
from __future__ import annotations

from functools import partial
from typing import Any

import gc
import os
import tempfile

import jax
import jax.numpy as jnp
import numpy as np
from jax.sharding import Mesh, PartitionSpec as P


PACK_SIZE = 128


def make_mesh(devices=None):
    if devices is None:
        devices = jax.devices()
    return Mesh(np.asarray(devices), axis_names=("cores",))


def _topk_smallest(values, k):
    neg_v, idx = jax.lax.top_k(-values, k)
    return -neg_v, idx


def _build_step_body_v_only(
    v_params,
    all_moves, V0, hash_vec, V0_hash,
    B_local, world_size, K_per_peer, n_gen, state_size,
    dtype, internal_bs,
):
    """Per-shard step body for V-only beam search."""
    BIG_F32 = jnp.float32(1e9)
    aB_local = K_per_peer * world_size

    def _chunked_apply(params, x, chunk_size):
        n, S = x.shape
        n_chunks = n // chunk_size
        chunks = x.reshape(n_chunks, chunk_size, S)
        def _scan_fn(_, chunk):
            return _, model_apply(params, chunk, dtype=dtype)
        _, outs = jax.lax.scan(_scan_fn, None, chunks)
        if outs.ndim == 2:
            return outs.reshape(n)
        return outs.reshape(n, -1)

    def forward_v(params, x):
        return _chunked_apply(params, x, internal_bs).astype(jnp.float32)

    # Precomputed per-child indexing (static).
    n_total = B_local * n_gen
    flat_idx = jnp.arange(n_total, dtype=jnp.int32)
    parent_local_per_child = (flat_idx // n_gen).astype(jnp.int32)
    move_per_child = (flat_idx % n_gen).astype(jnp.int8)
    arange_aB = jnp.arange(aB_local, dtype=jnp.int32)
    sender_rank_per_recv = (arange_aB // K_per_peer).astype(jnp.int8)

    def beam_step_local(states, tree_parent_local, tree_parent_rank, tree_move,
                       min_v_log, found_step, found_pos_local, found_pos_rank,
                       verify_state, j):
        # shard_map keeps sharded axes with local size; squeeze leading shard axis.
        states = states[0]
        tree_parent_local = tree_parent_local[0]
        tree_parent_rank = tree_parent_rank[0]
        tree_move = tree_move[0]
        min_v_log = min_v_log[0]
        found_step = found_step[0]
        found_pos_local = found_pos_local[0]
        found_pos_rank = found_pos_rank[0]
        verify_state = verify_state[0]
        rank_int = jax.lax.axis_index("cores").astype(jnp.int32)

        # 1. Generate all children of owned states.
        neighbors = states[:, all_moves].reshape(-1, state_size)  # (n_total, S)

        # 2. V on ALL children (the V-only step — no qshort prefilter).
        child_v = forward_v(v_params, neighbors)  # (n_total,) float32

        # 3. Hash + owner per child.
        h = jnp.sum(neighbors.astype(jnp.int64) * hash_vec, axis=1)
        owner = (h % jnp.int64(world_size)).astype(jnp.int32)

        # 4. Per-owner top-K_per_peer by V score. Loop over owners is unrolled.
        send_buckets = jnp.zeros((world_size, K_per_peer, PACK_SIZE), dtype=jnp.uint8)
        for S in range(world_size):
            mask_S = (owner == S)
            score_for_S = jnp.where(mask_S, child_v, BIG_F32)
            top_v_S, top_idx_S = _topk_smallest(score_for_S, K_per_peer)
            is_pad_S = top_v_S >= (BIG_F32 * 0.5)

            sel_states = neighbors[top_idx_S]
            sel_parent_local = parent_local_per_child[top_idx_S]
            sel_move = move_per_child[top_idx_S]

            zero_state = jnp.zeros((K_per_peer, state_size), dtype=jnp.uint8)
            zero_int32 = jnp.zeros(K_per_peer, dtype=jnp.int32)
            zero_int8 = jnp.zeros(K_per_peer, dtype=jnp.int8)

            sel_states_u8 = jnp.where(is_pad_S[:, None], zero_state, sel_states.astype(jnp.uint8))
            sel_parent_local_z = jnp.where(is_pad_S, zero_int32, sel_parent_local)
            sel_move_z = jnp.where(is_pad_S, zero_int8, sel_move)

            bucket = jnp.zeros((K_per_peer, PACK_SIZE), dtype=jnp.uint8)
            bucket = bucket.at[:, 0:120].set(sel_states_u8)
            bucket = bucket.at[:, 120].set((sel_parent_local_z & 0xFF).astype(jnp.uint8))
            bucket = bucket.at[:, 121].set(((sel_parent_local_z >> 8) & 0xFF).astype(jnp.uint8))
            bucket = bucket.at[:, 122].set(((sel_parent_local_z >> 16) & 0xFF).astype(jnp.uint8))
            bucket = bucket.at[:, 123].set(((sel_parent_local_z >> 24) & 0xFF).astype(jnp.uint8))
            bucket = bucket.at[:, 124].set(sel_move_z.astype(jnp.uint8))
            send_buckets = send_buckets.at[S].set(bucket)

        # 5. all_to_all: bucket S from each sender goes to rank S.
        recv_buckets = jax.lax.all_to_all(
            send_buckets, axis_name="cores",
            split_axis=0, concat_axis=0, tiled=True,
        )

        # 6. Unpack received candidates.
        recv_flat = recv_buckets.reshape(-1, PACK_SIZE)
        recv_states_u8 = recv_flat[:, 0:120]
        recv_states = recv_states_u8.astype(jnp.int8)
        recv_parent_local = (
            recv_flat[:, 120].astype(jnp.int32)
            | (recv_flat[:, 121].astype(jnp.int32) << 8)
            | (recv_flat[:, 122].astype(jnp.int32) << 16)
            | (recv_flat[:, 123].astype(jnp.int32) << 24)
        )
        recv_move = recv_flat[:, 124].astype(jnp.int8)
        recv_sender_rank = sender_rank_per_recv

        # Padding detection: real states sum to 7140, padding sums to 0.
        recv_state_sum = jnp.sum(recv_states.astype(jnp.int32), axis=1)
        is_padding = (recv_state_sum == 0)

        # In-rank dedup.
        recv_h = jnp.sum(recv_states.astype(jnp.int64) * hash_vec, axis=1)
        sort_h = jnp.sort(recv_h)
        sort_idx = jnp.argsort(recv_h)
        is_dup_sorted = jnp.concatenate([
            jnp.zeros(1, dtype=jnp.bool_),
            sort_h[1:] == sort_h[:-1],
        ])
        restore = jnp.argsort(sort_idx)
        dup_mask = is_dup_sorted[restore]

        # 7. Re-run V on received states (necessary because we didn't pack V
        # scores into the bucket — keeps PACK_SIZE=128, same as qshort body).
        # aB_local ≈ 2 * B_local is small vs. n_total = 24 * B_local so the
        # extra compute is ~8%.
        recv_v = forward_v(v_params, recv_states)
        recv_v_masked = jnp.where(dup_mask | is_padding, BIG_F32, recv_v)

        top_v_keep, keep_idx = _topk_smallest(recv_v_masked, B_local)

        chosen_states = recv_states[keep_idx]
        chosen_parent_local = recv_parent_local[keep_idx]
        chosen_parent_rank = recv_sender_rank[keep_idx]
        chosen_move = recv_move[keep_idx]
        chosen_h = recv_h[keep_idx]
        chosen_state_sum = jnp.sum(chosen_states.astype(jnp.int32), axis=1)
        chosen_is_real = (chosen_state_sum != 0)

        new_tree_parent_local = tree_parent_local.at[j].set(chosen_parent_local)
        new_tree_parent_rank = tree_parent_rank.at[j].set(chosen_parent_rank)
        new_tree_move = tree_move.at[j].set(chosen_move)
        new_min_v_log = min_v_log.at[j].set(top_v_keep[0])

        # V0 detection (per-rank; cross-rank reduce REMOVED -- host-side instead).
        eq_v0 = (chosen_h == V0_hash) & chosen_is_real
        any_hit = jnp.any(eq_v0)
        pos_hit = jnp.argmax(eq_v0.astype(jnp.int32)).astype(jnp.int32)
        is_first_hit = (found_step == -1) & any_hit
        new_found_step = jnp.where(is_first_hit, j, found_step)
        new_found_pos_local = jnp.where(is_first_hit, pos_hit, found_pos_local)
        new_found_pos_rank = jnp.where(is_first_hit, rank_int, found_pos_rank)
        candidate_state = chosen_states[pos_hit]
        new_verify_state = jnp.where(is_first_hit, candidate_state, verify_state)

        return (chosen_states[None, :, :],
                new_tree_parent_local[None, :, :],
                new_tree_parent_rank[None, :, :],
                new_tree_move[None, :, :],
                new_min_v_log[None, :],
                new_found_step[None],
                new_found_pos_local[None],
                new_found_pos_rank[None],
                new_verify_state[None, :])

    return beam_step_local


def beam_solve_v_only_spmd(
    init_state_list: list[int],
    v_params,
    all_moves: jnp.ndarray,
    V0: jnp.ndarray,
    hash_vec: jnp.ndarray,
    mesh: Mesh,
    B_local: int,
    K_per_peer: int,
    n_gen: int = 24,
    state_size: int = 120,
    num_steps: int = 120,
    dtype=jnp.bfloat16,
    internal_bs: int = 32768,
) -> dict[str, Any]:
    """High-level V-only SPMD beam solver."""
    devices = mesh.devices.flatten()
    world_size = len(devices)
    init_state = np.asarray(init_state_list, dtype=np.int8)
    if np.array_equal(init_state, np.asarray(V0)):
        return {"found": True, "path_len": 0, "path_idx": [], "found_step": -1, "wall_s": 0.0}

    # Step 0: V on 24 first-move children (always unchunked).
    init_dev = jnp.asarray(init_state)
    states_seed = jnp.expand_dims(init_dev, 0)
    neighbors0 = states_seed[:, all_moves].reshape(-1, state_size)
    values0 = model_apply(v_params, neighbors0, dtype=dtype).astype(jnp.float32)
    k0 = min(B_local, n_gen)
    _, top_idx0 = _topk_smallest(values0, k0)
    chosen0 = neighbors0[top_idx0]
    if k0 < B_local:
        pad = jnp.broadcast_to(chosen0[-1:], (B_local - k0, state_size))
        states0 = jnp.concatenate([chosen0, pad], axis=0)
    else:
        states0 = chosen0[:B_local]

    states_global = jnp.broadcast_to(states0[None, :, :], (world_size, B_local, state_size))
    tree_parent_local = jnp.zeros((world_size, num_steps, B_local), dtype=jnp.int32)
    tree_parent_rank = jnp.zeros((world_size, num_steps, B_local), dtype=jnp.int8)
    tree_move = jnp.full((world_size, num_steps, B_local), -1, dtype=jnp.int8)

    top_idx0_i8 = top_idx0.astype(jnp.int8)
    if k0 < B_local:
        last_move = top_idx0_i8[k0 - 1]
        pad_moves = jnp.broadcast_to(last_move, (B_local - k0,))
        move0_full = jnp.concatenate([top_idx0_i8, pad_moves])
    else:
        move0_full = top_idx0_i8[:B_local]
    move0_full = jnp.broadcast_to(move0_full[None, :], (world_size, B_local))
    tree_move = tree_move.at[:, 0, :].set(move0_full)

    min_v_log = jnp.full((world_size, num_steps), 1e6, dtype=jnp.float32)
    found_step = jnp.full((world_size,), -1, dtype=jnp.int32)
    found_pos_local = jnp.full((world_size,), -1, dtype=jnp.int32)
    found_pos_rank = jnp.full((world_size,), -1, dtype=jnp.int32)
    verify_state = jnp.zeros((world_size, state_size), dtype=jnp.int8)

    eq0 = jnp.all(states0 == V0, axis=1)
    any0 = jnp.any(eq0)
    pos0 = jnp.argmax(eq0.astype(jnp.int32)).astype(jnp.int32)
    if bool(any0):
        pos0_h = int(pos0)
        if pos0_h < k0:
            seed_move = int(top_idx0[pos0_h])
        else:
            seed_move = int(top_idx0[k0 - 1])
        return {"found": True, "path_len": 1, "path_idx": [seed_move],
                "found_step": 0, "wall_s": 0.0}

    V0_hash_host = int(np.sum(np.asarray(V0).astype(np.int64) * np.asarray(hash_vec)))

    step_body = _build_step_body_v_only(
        v_params, all_moves, V0, hash_vec, jnp.int64(V0_hash_host),
        B_local, world_size, K_per_peer, n_gen, state_size,
        dtype, int(internal_bs),
    )

    @jax.jit
    def step_fn(states, tp_local, tp_rank, tmove, mv_log, fs, fpl, fpr, vstate, j_arr):
        try:
            from jax.experimental.shard_map import shard_map
        except ImportError:
            from jax import shard_map
        return shard_map(
            step_body,
            mesh=mesh,
            in_specs=(P("cores"), P("cores"), P("cores"), P("cores"),
                      P("cores"), P("cores"), P("cores"), P("cores"),
                      P("cores"), P()),
            out_specs=(P("cores"), P("cores"), P("cores"), P("cores"),
                       P("cores"), P("cores"), P("cores"), P("cores"),
                       P("cores")),
        )(states, tp_local, tp_rank, tmove, mv_log, fs, fpl, fpr, vstate, j_arr)

    from jax.sharding import NamedSharding
    sharding_states = NamedSharding(mesh, P("cores"))
    sharding_scalar = NamedSharding(mesh, P("cores"))

    states_d = jax.device_put(states_global, sharding_states)
    tp_local_d = jax.device_put(tree_parent_local, sharding_states)
    tp_rank_d = jax.device_put(tree_parent_rank, sharding_states)
    tmove_d = jax.device_put(tree_move, sharding_states)
    mv_log_d = jax.device_put(min_v_log, sharding_scalar)
    fs_d = jax.device_put(found_step, sharding_scalar)
    fpl_d = jax.device_put(found_pos_local, sharding_scalar)
    fpr_d = jax.device_put(found_pos_rank, sharding_scalar)
    vstate_d = jax.device_put(verify_state, sharding_states)

    import time
    t_start = time.time()
    first_iter_t = None

    for j in range(1, num_steps):
        t_iter = time.time()
        j_arr = jnp.int32(j)
        (states_d, tp_local_d, tp_rank_d, tmove_d,
         mv_log_d, fs_d, fpl_d, fpr_d, vstate_d) = step_fn(
            states_d, tp_local_d, tp_rank_d, tmove_d,
            mv_log_d, fs_d, fpl_d, fpr_d, vstate_d, j_arr,
        )
        if first_iter_t is None:
            jax.block_until_ready(fs_d)
            first_iter_t = time.time() - t_iter

    jax.block_until_ready(fs_d)

    # Host-side cross-rank winner selection.
    fs_per_rank = np.asarray(fs_d)
    fpl_per_rank = np.asarray(fpl_d)
    fpr_per_rank = np.asarray(fpr_d)

    INT_MAX = 2 ** 30
    fs_signed = np.where(fs_per_rank >= 0, fs_per_rank, INT_MAX)
    global_min_step = int(fs_signed.min())
    if global_min_step >= INT_MAX:
        return {"found": False, "path_len": 0, "path_idx": [],
                "found_step": -1, "wall_s": time.time() - t_start,
                "first_iter_s": first_iter_t}

    winner_ranks = np.where(fs_signed == global_min_step)[0]
    winner_rank = int(winner_ranks[0])
    fs_h = int(fs_per_rank[winner_rank])
    fpl_h = int(fpl_per_rank[winner_rank])
    fpr_h = winner_rank

    tp_local_h = np.asarray(tp_local_d)
    tp_rank_h = np.asarray(tp_rank_d)
    tmove_h = np.asarray(tmove_d)

    path_idx = []
    cur_rank = fpr_h
    cur_pos = fpl_h
    for j in range(fs_h, -1, -1):
        m = int(tmove_h[cur_rank, j, cur_pos])
        if m >= 0:
            path_idx.append(m)
        if j > 0:
            new_rank = int(tp_rank_h[cur_rank, j, cur_pos])
            new_pos = int(tp_local_h[cur_rank, j, cur_pos])
            cur_rank = new_rank
            cur_pos = new_pos
    path_idx.reverse()
    while path_idx and path_idx[0] < 0:
        path_idx.pop(0)

    return {
        "found": True,
        "path_len": len(path_idx),
        "path_idx": path_idx,
        "found_step": fs_h,
        "found_pos_local": fpl_h,
        "found_pos_rank": fpr_h,
        "wall_s": time.time() - t_start,
        "first_iter_s": first_iter_t,
    }


# =============================================================================
# Phase A+B variant: packed uint32 backpointers, host memmap tree, early stop.
# =============================================================================

def _build_step_body_v_only_packed(
    v_params,
    all_moves, V0, hash_vec, V0_hash,
    B_local, world_size, K_per_peer, n_gen, state_size,
    dtype, internal_bs,
    pack_v_score=False,
    owner_hash_vec=None,
):
    """Per-shard V-only step body, Phase A+B (+ optional D) variant.

    `pack_v_score` (Phase D): if True, pack the send-side bf16 V score into
    bucket bytes 125-126 and skip the receive-side forward_v re-run. Saves
    ~8% V compute (the receive-side forward on aB_local states). Quality:
    bf16 ordering may tie-break differently than fp32 -- expect path-length
    drift of 0-2 moves per pid.

    Differences from `_build_step_body_v_only`:
      * Phase A: precomputed parent_local_per_child / move_per_child arrays
        removed. Per-owner parent/move derived from `top_idx_S` via integer
        div/mod after the topk. Saves ~B_local*n_gen*(4+1) bytes per rank
        (180 MB at B_local=4M).
      * Phase B: tree_* tensors removed from the carry. The body now emits a
        single packed uint32 backpointer per beam slot (shape (B_local,)):
          bits  0..22: parent_local (supports up to 8,388,608 = 64M global)
          bits 23..25: parent_rank  (0..7)
          bits 26..30: move         (0..23, 5 bits)
          bit      31: unused (could later be a "valid" flag)

    Body outputs (in shard-local shape, with leading rank axis re-added):
      (chosen_states, packed_backptr, min_v_log, found_step,
       found_pos_local, found_pos_rank, verify_state)

    The wrapper writes packed_backptr into a host np.memmap each step.
    Walkback unpacks records and follows (parent_rank, parent_local) chains.
    """
    BIG_F32 = jnp.float32(1e9)
    aB_local = K_per_peer * world_size

    def _chunked_apply(params, x, chunk_size):
        n, S = x.shape
        n_chunks = n // chunk_size
        chunks = x.reshape(n_chunks, chunk_size, S)
        def _scan_fn(_, chunk):
            return _, model_apply(params, chunk, dtype=dtype)
        _, outs = jax.lax.scan(_scan_fn, None, chunks)
        if outs.ndim == 2:
            return outs.reshape(n)
        return outs.reshape(n, -1)

    def forward_v(params, x):
        return _chunked_apply(params, x, internal_bs).astype(jnp.float32)

    # Phase A: full per-child index arrays dropped. Only receive-side
    # sender-rank index remains (computed once, ~aB_local bytes int8).
    arange_aB = jnp.arange(aB_local, dtype=jnp.int32)
    sender_rank_per_recv = (arange_aB // K_per_peer).astype(jnp.int8)

    def beam_step_local(states, min_v_log, found_step, found_pos_local,
                        found_pos_rank, verify_state, j):
        # Squeeze leading shard axis (shard_map gives each rank shape (1, ...)).
        states = states[0]
        min_v_log = min_v_log[0]
        found_step = found_step[0]
        found_pos_local = found_pos_local[0]
        found_pos_rank = found_pos_rank[0]
        verify_state = verify_state[0]
        rank_int = jax.lax.axis_index("cores").astype(jnp.int32)

        # 1. Generate all children of owned states.
        neighbors = states[:, all_moves].reshape(-1, state_size)

        # 2. V on ALL children (chunked via lax.scan).
        child_v = forward_v(v_params, neighbors)

        # 3. Owner hash. Owner routing only needs uniform partitioning, not
        # collision resistance -- use cheaper uint32 hash if provided.
        # world_size is a power of 2 (8 on v5e-8) so AND-mask is cheaper than mod.
        if owner_hash_vec is not None:
            h_owner = jnp.sum(neighbors.astype(jnp.uint32) * owner_hash_vec, axis=1)
            owner = (h_owner & jnp.uint32(world_size - 1)).astype(jnp.int32)
        else:
            h = jnp.sum(neighbors.astype(jnp.int64) * hash_vec, axis=1)
            owner = (h % jnp.int64(world_size)).astype(jnp.int32)

        # 4. Per-owner top-K_per_peer by V score (loop unrolled in JIT).
        send_buckets = jnp.zeros((world_size, K_per_peer, PACK_SIZE), dtype=jnp.uint8)
        for S in range(world_size):
            mask_S = (owner == S)
            score_for_S = jnp.where(mask_S, child_v, BIG_F32)
            top_v_S, top_idx_S = _topk_smallest(score_for_S, K_per_peer)
            is_pad_S = top_v_S >= (BIG_F32 * 0.5)

            sel_states = neighbors[top_idx_S]
            # Phase A: derive parent_local / move from top_idx_S on demand.
            sel_parent_local = (top_idx_S // n_gen).astype(jnp.int32)
            sel_move = (top_idx_S % n_gen).astype(jnp.int8)

            zero_state = jnp.zeros((K_per_peer, state_size), dtype=jnp.uint8)
            zero_int32 = jnp.zeros(K_per_peer, dtype=jnp.int32)
            zero_int8 = jnp.zeros(K_per_peer, dtype=jnp.int8)

            sel_states_u8 = jnp.where(is_pad_S[:, None], zero_state, sel_states.astype(jnp.uint8))
            sel_parent_local_z = jnp.where(is_pad_S, zero_int32, sel_parent_local)
            sel_move_z = jnp.where(is_pad_S, zero_int8, sel_move)

            bucket = jnp.zeros((K_per_peer, PACK_SIZE), dtype=jnp.uint8)
            bucket = bucket.at[:, 0:120].set(sel_states_u8)
            bucket = bucket.at[:, 120].set((sel_parent_local_z & 0xFF).astype(jnp.uint8))
            bucket = bucket.at[:, 121].set(((sel_parent_local_z >> 8) & 0xFF).astype(jnp.uint8))
            bucket = bucket.at[:, 122].set(((sel_parent_local_z >> 16) & 0xFF).astype(jnp.uint8))
            bucket = bucket.at[:, 123].set(((sel_parent_local_z >> 24) & 0xFF).astype(jnp.uint8))
            bucket = bucket.at[:, 124].set(sel_move_z.astype(jnp.uint8))
            if pack_v_score:
                # Phase D: pack bf16 V score into bytes 125-126.
                top_v_S_bf16 = top_v_S.astype(jnp.bfloat16)
                top_v_S_u16 = jax.lax.bitcast_convert_type(top_v_S_bf16, jnp.uint16)
                bucket = bucket.at[:, 125].set((top_v_S_u16 & 0xFF).astype(jnp.uint8))
                bucket = bucket.at[:, 126].set(((top_v_S_u16 >> 8) & 0xFF).astype(jnp.uint8))
            send_buckets = send_buckets.at[S].set(bucket)

        # 5. all_to_all: bucket S from each sender goes to rank S.
        recv_buckets = jax.lax.all_to_all(
            send_buckets, axis_name="cores",
            split_axis=0, concat_axis=0, tiled=True,
        )

        # 6. Unpack received candidates.
        recv_flat = recv_buckets.reshape(-1, PACK_SIZE)
        recv_states_u8 = recv_flat[:, 0:120]
        recv_states = recv_states_u8.astype(jnp.int8)
        recv_parent_local = (
            recv_flat[:, 120].astype(jnp.int32)
            | (recv_flat[:, 121].astype(jnp.int32) << 8)
            | (recv_flat[:, 122].astype(jnp.int32) << 16)
            | (recv_flat[:, 123].astype(jnp.int32) << 24)
        )
        recv_move = recv_flat[:, 124].astype(jnp.int8)
        recv_sender_rank = sender_rank_per_recv

        # Padding detection: real states sum to 7140, padding sums to 0.
        recv_state_sum = jnp.sum(recv_states.astype(jnp.int32), axis=1)
        is_padding = (recv_state_sum == 0)

        # In-rank dedup via sorted-order traversal (one argsort + gathers).
        # Top-k doesn't care whether candidates are in original receive order,
        # so we process everything in sorted layout and drop the second argsort
        # (the inverse-permutation "restore" that the prior version needed).
        recv_h = jnp.sum(recv_states.astype(jnp.int64) * hash_vec, axis=1)
        sort_idx = jnp.argsort(recv_h)
        sorted_h = recv_h[sort_idx]
        sorted_states = recv_states[sort_idx]
        sorted_parent_local = recv_parent_local[sort_idx]
        sorted_move = recv_move[sort_idx]
        sorted_sender_rank = recv_sender_rank[sort_idx]
        sorted_is_padding = is_padding[sort_idx]
        is_dup_sorted = jnp.concatenate([
            jnp.zeros(1, dtype=jnp.bool_),
            sorted_h[1:] == sorted_h[:-1],
        ])

        # 7. Re-run V on received states -- OR unpack the packed score (Phase D).
        if pack_v_score:
            recv_v_u16 = (
                recv_flat[:, 125].astype(jnp.uint16)
                | (recv_flat[:, 126].astype(jnp.uint16) << jnp.uint16(8))
            )
            recv_v_bf16 = jax.lax.bitcast_convert_type(recv_v_u16, jnp.bfloat16)
            recv_v = recv_v_bf16.astype(jnp.float32)
        else:
            recv_v = forward_v(v_params, recv_states)
        sorted_v = recv_v[sort_idx]
        sorted_v_masked = jnp.where(is_dup_sorted | sorted_is_padding, BIG_F32, sorted_v)

        top_v_keep, keep_sorted_idx = _topk_smallest(sorted_v_masked, B_local)

        chosen_states = sorted_states[keep_sorted_idx]
        chosen_parent_local = sorted_parent_local[keep_sorted_idx]
        chosen_parent_rank = sorted_sender_rank[keep_sorted_idx]
        chosen_move = sorted_move[keep_sorted_idx]
        chosen_h = sorted_h[keep_sorted_idx]
        chosen_state_sum = jnp.sum(chosen_states.astype(jnp.int32), axis=1)
        chosen_is_real = (chosen_state_sum != 0)

        # Phase B: pack backpointer into a single uint32 per beam slot.
        packed_backptr = (
            chosen_parent_local.astype(jnp.uint32)
            | (chosen_parent_rank.astype(jnp.uint32) << jnp.uint32(23))
            | (chosen_move.astype(jnp.uint32) << jnp.uint32(26))
        )

        new_min_v_log = min_v_log.at[j].set(top_v_keep[0])

        # V0 detection (per-rank; cross-rank reduce REMOVED -- host-side instead).
        eq_v0 = (chosen_h == V0_hash) & chosen_is_real
        any_hit = jnp.any(eq_v0)
        pos_hit = jnp.argmax(eq_v0.astype(jnp.int32)).astype(jnp.int32)
        is_first_hit = (found_step == -1) & any_hit
        new_found_step = jnp.where(is_first_hit, j, found_step)
        new_found_pos_local = jnp.where(is_first_hit, pos_hit, found_pos_local)
        new_found_pos_rank = jnp.where(is_first_hit, rank_int, found_pos_rank)
        candidate_state = chosen_states[pos_hit]
        new_verify_state = jnp.where(is_first_hit, candidate_state, verify_state)

        return (chosen_states[None, :, :],
                packed_backptr[None, :],
                new_min_v_log[None, :],
                new_found_step[None],
                new_found_pos_local[None],
                new_found_pos_rank[None],
                new_verify_state[None, :])

    return beam_step_local


# -----------------------------------------------------------------------------
# Phase C: streaming child generation via lax.scan over parent chunks.
# Removes the 11.5 GiB neighbors materialization at B_local=4M (32M global).
# Same packed-backpointer output as the non-streaming body, so the wrapper
# can dispatch to either body via the `parent_chunk` parameter.
# -----------------------------------------------------------------------------

def _build_step_body_v_only_packed_streaming(
    v_params,
    all_moves, V0, hash_vec, V0_hash,
    B_local, world_size, K_per_peer, n_gen, state_size,
    dtype, internal_bs,
    parent_chunk,
    pack_v_score=False,
    owner_hash_vec=None,
):
    """Per-shard V-only step body, Phase A+B+C (+ optional D): streamed.

    `pack_v_score` (Phase D): same as in the non-streaming body -- pack send-side
    score into bytes 125-126, skip receive-side forward_v. Quality budget:
    0-2 moves per pid from bf16 tie-break drift.

    Replaces `_build_step_body_v_only_packed`'s full-neighbors materialization
    (B_local * n_gen * state_size bytes; 11.5 GiB at B_local=4M) with a
    `lax.scan` over parent chunks. Per chunk:
      * generate (parent_chunk * n_gen, state_size) children
      * run V on the chunk's children (chunked internally by internal_bs)
      * route to owner buckets, merge with per-owner running top-K_per_peer
    Final per-rank send_buckets = (world_size, K_per_peer, PACK_SIZE) uint8.

    Everything downstream of the all_to_all is identical to the non-streaming
    body. Backpointer is the same packed uint32 format.

    `parent_chunk` MUST divide B_local. Typical value at 32M: 65536.
    """
    BIG_F32 = jnp.float32(1e9)
    aB_local = K_per_peer * world_size
    assert B_local % parent_chunk == 0, (
        f"parent_chunk {parent_chunk} must divide B_local {B_local}"
    )
    n_chunks = B_local // parent_chunk
    chunk_n = parent_chunk * n_gen

    def _chunked_apply(params, x, chunk_size):
        n, S = x.shape
        n_chunks_inner = n // chunk_size
        chunks = x.reshape(n_chunks_inner, chunk_size, S)
        def _scan_fn(_, c):
            return _, model_apply(params, c, dtype=dtype)
        _, outs = jax.lax.scan(_scan_fn, None, chunks)
        if outs.ndim == 2:
            return outs.reshape(n)
        return outs.reshape(n, -1)

    def forward_v(params, x):
        return _chunked_apply(params, x, internal_bs).astype(jnp.float32)

    # Receive-side sender_rank index (static, small).
    arange_aB = jnp.arange(aB_local, dtype=jnp.int32)
    sender_rank_per_recv = (arange_aB // K_per_peer).astype(jnp.int8)

    def beam_step_local(states, min_v_log, found_step, found_pos_local,
                        found_pos_rank, verify_state, j):
        # Squeeze leading shard axis.
        states = states[0]
        min_v_log = min_v_log[0]
        found_step = found_step[0]
        found_pos_local = found_pos_local[0]
        found_pos_rank = found_pos_rank[0]
        verify_state = verify_state[0]
        rank_int = jax.lax.axis_index("cores").astype(jnp.int32)

        # Streaming scan carry: per-owner running top-K_per_peer.
        # Initial: all-BIG_F32 scores, all-zero packs. Any real child wins.
        # Inside shard_map's manual mode, lax.scan requires the carry to be
        # 'varying' along the sharded axis. Constants from jnp.full/jnp.zeros
        # are replicated by default; pcast them to 'varying' so the
        # carry-in/carry-out types match.
        init_top_scores = jax.lax.pcast(
            jnp.full((world_size, K_per_peer), BIG_F32, dtype=jnp.float32),
            ("cores",), to="varying",
        )
        init_top_pack = jax.lax.pcast(
            jnp.zeros((world_size, K_per_peer, PACK_SIZE), dtype=jnp.uint8),
            ("cores",), to="varying",
        )

        def chunk_body(carry, chunk_i):
            top_scores, top_pack = carry
            parent_start = chunk_i * jnp.int32(parent_chunk)
            states_chunk = jax.lax.dynamic_slice(
                states, (parent_start, jnp.int32(0)),
                (parent_chunk, state_size),
            )
            children = states_chunk[:, all_moves].reshape(-1, state_size)  # (chunk_n, S)

            # V on chunk's children (chunked internally via _chunked_apply).
            child_v = forward_v(v_params, children)

            # Owner partition. Owner routing only needs a uniform 0..world_size-1
            # bucket; use cheaper uint32 hash if provided. world_size is a power
            # of 2 (8 on v5e-8), so AND-mask is cheaper than mod.
            if owner_hash_vec is not None:
                h_owner = jnp.sum(children.astype(jnp.uint32) * owner_hash_vec, axis=1)
                owner = (h_owner & jnp.uint32(world_size - 1)).astype(jnp.int32)
            else:
                h = jnp.sum(children.astype(jnp.int64) * hash_vec, axis=1)
                owner = (h % jnp.int64(world_size)).astype(jnp.int32)

            new_top_scores = top_scores
            new_top_pack = top_pack
            for dest in range(world_size):
                # Mask children not owned by `dest`.
                masked_scores = jnp.where(owner == dest, child_v, BIG_F32)
                # Merge running top with this chunk's masked candidates.
                merged_scores = jnp.concatenate(
                    [new_top_scores[dest], masked_scores], axis=0,
                )  # (K_per_peer + chunk_n,) float32
                top_v_new, keep = _topk_smallest(merged_scores, K_per_peer)

                # Each kept slot is either from old (keep < K_per_peer) or new.
                from_old = keep < K_per_peer
                old_idx = jnp.clip(keep, 0, K_per_peer - 1)
                new_idx = jnp.clip(keep - K_per_peer, 0, chunk_n - 1)

                # Old pack: previous top_pack[dest] indexed by old_idx.
                old_pack = new_top_pack[dest][old_idx]  # (K_per_peer, PACK_SIZE)

                # New pack: pack chunk children at new_idx.
                sel_states = children[new_idx]  # (K_per_peer, state_size) int8
                sel_parent_local = (parent_start + (new_idx // n_gen)).astype(jnp.int32)
                sel_move = (new_idx % n_gen).astype(jnp.int8)

                # Padding marker (a kept slot is pad iff its score is BIG_F32).
                is_pad = top_v_new >= (BIG_F32 * 0.5)
                zero_state = jnp.zeros((K_per_peer, state_size), dtype=jnp.uint8)
                zero_int32 = jnp.zeros(K_per_peer, dtype=jnp.int32)
                zero_int8 = jnp.zeros(K_per_peer, dtype=jnp.int8)
                sel_states_u8 = jnp.where(is_pad[:, None], zero_state, sel_states.astype(jnp.uint8))
                sel_parent_local_z = jnp.where(is_pad, zero_int32, sel_parent_local)
                sel_move_z = jnp.where(is_pad, zero_int8, sel_move)

                new_pack = jnp.zeros((K_per_peer, PACK_SIZE), dtype=jnp.uint8)
                new_pack = new_pack.at[:, 0:120].set(sel_states_u8)
                new_pack = new_pack.at[:, 120].set((sel_parent_local_z & 0xFF).astype(jnp.uint8))
                new_pack = new_pack.at[:, 121].set(((sel_parent_local_z >> 8) & 0xFF).astype(jnp.uint8))
                new_pack = new_pack.at[:, 122].set(((sel_parent_local_z >> 16) & 0xFF).astype(jnp.uint8))
                new_pack = new_pack.at[:, 123].set(((sel_parent_local_z >> 24) & 0xFF).astype(jnp.uint8))
                new_pack = new_pack.at[:, 124].set(sel_move_z.astype(jnp.uint8))

                # Select between old_pack and new_pack per kept slot.
                merged_pack = jnp.where(from_old[:, None], old_pack, new_pack)

                if pack_v_score:
                    # Phase D: pack bf16 score (post-topk) into bytes 125-126.
                    # top_v_new[i] is the right score regardless of from_old/new.
                    top_v_bf16 = top_v_new.astype(jnp.bfloat16)
                    top_v_u16 = jax.lax.bitcast_convert_type(top_v_bf16, jnp.uint16)
                    merged_pack = merged_pack.at[:, 125].set((top_v_u16 & 0xFF).astype(jnp.uint8))
                    merged_pack = merged_pack.at[:, 126].set(((top_v_u16 >> 8) & 0xFF).astype(jnp.uint8))

                new_top_scores = new_top_scores.at[dest].set(top_v_new)
                new_top_pack = new_top_pack.at[dest].set(merged_pack)

            return (new_top_scores, new_top_pack), None

        (_, final_top_pack), _ = jax.lax.scan(
            chunk_body, (init_top_scores, init_top_pack),
            jnp.arange(n_chunks, dtype=jnp.int32),
        )

        send_buckets = final_top_pack  # (world_size, K_per_peer, PACK_SIZE)

        # all_to_all + receive side (identical to non-streaming body).
        recv_buckets = jax.lax.all_to_all(
            send_buckets, axis_name="cores",
            split_axis=0, concat_axis=0, tiled=True,
        )

        recv_flat = recv_buckets.reshape(-1, PACK_SIZE)
        recv_states_u8 = recv_flat[:, 0:120]
        recv_states = recv_states_u8.astype(jnp.int8)
        recv_parent_local = (
            recv_flat[:, 120].astype(jnp.int32)
            | (recv_flat[:, 121].astype(jnp.int32) << 8)
            | (recv_flat[:, 122].astype(jnp.int32) << 16)
            | (recv_flat[:, 123].astype(jnp.int32) << 24)
        )
        recv_move = recv_flat[:, 124].astype(jnp.int8)
        recv_sender_rank = sender_rank_per_recv

        recv_state_sum = jnp.sum(recv_states.astype(jnp.int32), axis=1)
        is_padding = (recv_state_sum == 0)

        # In-rank dedup via sorted-order traversal (one argsort + gathers).
        recv_h = jnp.sum(recv_states.astype(jnp.int64) * hash_vec, axis=1)
        sort_idx = jnp.argsort(recv_h)
        sorted_h = recv_h[sort_idx]
        sorted_states = recv_states[sort_idx]
        sorted_parent_local = recv_parent_local[sort_idx]
        sorted_move = recv_move[sort_idx]
        sorted_sender_rank = recv_sender_rank[sort_idx]
        sorted_is_padding = is_padding[sort_idx]
        is_dup_sorted = jnp.concatenate([
            jnp.zeros(1, dtype=jnp.bool_),
            sorted_h[1:] == sorted_h[:-1],
        ])

        # Re-run V on received states -- OR unpack the packed score (Phase D).
        if pack_v_score:
            recv_v_u16 = (
                recv_flat[:, 125].astype(jnp.uint16)
                | (recv_flat[:, 126].astype(jnp.uint16) << jnp.uint16(8))
            )
            recv_v_bf16 = jax.lax.bitcast_convert_type(recv_v_u16, jnp.bfloat16)
            recv_v = recv_v_bf16.astype(jnp.float32)
        else:
            recv_v = forward_v(v_params, recv_states)
        sorted_v = recv_v[sort_idx]
        sorted_v_masked = jnp.where(is_dup_sorted | sorted_is_padding, BIG_F32, sorted_v)

        top_v_keep, keep_sorted_idx = _topk_smallest(sorted_v_masked, B_local)

        chosen_states = sorted_states[keep_sorted_idx]
        chosen_parent_local = sorted_parent_local[keep_sorted_idx]
        chosen_parent_rank = sorted_sender_rank[keep_sorted_idx]
        chosen_move = sorted_move[keep_sorted_idx]
        chosen_h = sorted_h[keep_sorted_idx]
        chosen_state_sum = jnp.sum(chosen_states.astype(jnp.int32), axis=1)
        chosen_is_real = (chosen_state_sum != 0)

        packed_backptr = (
            chosen_parent_local.astype(jnp.uint32)
            | (chosen_parent_rank.astype(jnp.uint32) << jnp.uint32(23))
            | (chosen_move.astype(jnp.uint32) << jnp.uint32(26))
        )

        new_min_v_log = min_v_log.at[j].set(top_v_keep[0])

        eq_v0 = (chosen_h == V0_hash) & chosen_is_real
        any_hit = jnp.any(eq_v0)
        pos_hit = jnp.argmax(eq_v0.astype(jnp.int32)).astype(jnp.int32)
        is_first_hit = (found_step == -1) & any_hit
        new_found_step = jnp.where(is_first_hit, j, found_step)
        new_found_pos_local = jnp.where(is_first_hit, pos_hit, found_pos_local)
        new_found_pos_rank = jnp.where(is_first_hit, rank_int, found_pos_rank)
        candidate_state = chosen_states[pos_hit]
        new_verify_state = jnp.where(is_first_hit, candidate_state, verify_state)

        return (chosen_states[None, :, :],
                packed_backptr[None, :],
                new_min_v_log[None, :],
                new_found_step[None],
                new_found_pos_local[None],
                new_found_pos_rank[None],
                new_verify_state[None, :])

    return beam_step_local


def beam_solve_v_only_spmd_packed(
    init_state_list: list[int],
    v_params,
    all_moves: jnp.ndarray,
    V0: jnp.ndarray,
    hash_vec: jnp.ndarray,
    mesh: Mesh,
    B_local: int,
    K_per_peer: int,
    n_gen: int = 24,
    state_size: int = 120,
    num_steps: int = 120,
    dtype=jnp.bfloat16,
    internal_bs: int = 32768,
    tree_path: str | None = None,
    parent_chunk: int | None = None,
    pack_v_score: bool = False,
    progress_every: int = 0,
    owner_hash_vec: jnp.ndarray | None = None,
) -> dict[str, Any]:
    """V-only SPMD solver, Phase A+B (+ optional C streaming, optional D V-packing):
    packed host-memmap tree + early stop.

    `parent_chunk`:
      * None (default): use the non-streaming body (full neighbors materialized).
        Safe up to B_GLOBAL=16M on v5e-8 with Phase A+B.
      * int (e.g. 65536): use the Phase C streaming body. `parent_chunk` must
        divide B_local. Required for B_GLOBAL >= 32M on v5e-8.

    `pack_v_score` (Phase D):
      * False (default): receive side re-runs V on packed candidates.
      * True: send side packs the bf16 V score into bucket bytes 125-126;
        receive side unpacks and skips the V forward. Saves ~8% V compute.
        bf16 tie-breaks may shift path lengths by 0-2 moves per pid.

    Differences from `beam_solve_v_only_spmd`:
      * Per-step backpointer is a packed uint32 emitted by the body and
        written to a host np.memmap (shape (num_steps, world_size, B_local)).
        Saves ~world_size*num_steps*B_local*6 bytes of device HBM
        (1.44 GB at B_local=2M, 2.88 GB at B_local=4M).
      * Wrapper exits the iter loop as soon as any rank locally detects V0
        (saves both host writes and TPU compute on remaining steps).
      * step_fn re-enables `donate_argnums` for the 6-tensor carry. JAX
        aliases each donated input buffer to its same-shaped output, saving
        another ~1.5 GB of HBM at B_local=2M.
      * Tree file is auto-deleted in a `finally` block.

    Memmap default location: /kaggle/working (Kaggle TPU scratch, ~20 GB),
    else `tempfile.gettempdir()`. Caller may override via `tree_path`.
    File size = num_steps * world_size * B_local * 4 bytes
    (e.g. 7.68 GB at B_local=2M, 15.36 GB at B_local=4M; both fit Kaggle's
    /kaggle/working).
    """
    devices = mesh.devices.flatten()
    world_size = len(devices)
    init_state = np.asarray(init_state_list, dtype=np.int8)
    if np.array_equal(init_state, np.asarray(V0)):
        return {"found": True, "path_len": 0, "path_idx": [], "found_step": -1, "wall_s": 0.0}

    if progress_every:
        print(
            f"[solve] B_local={B_local:,} K_per_peer={K_per_peer:,} "
            f"parent_chunk={parent_chunk} internal_bs={internal_bs} "
            f"pack_v_score={pack_v_score} num_steps={num_steps}",
            flush=True,
        )

    # Step 0: V on 24 first-move children (same as non-packed variant).
    init_dev = jnp.asarray(init_state)
    states_seed = jnp.expand_dims(init_dev, 0)
    neighbors0 = states_seed[:, all_moves].reshape(-1, state_size)
    values0 = model_apply(v_params, neighbors0, dtype=dtype).astype(jnp.float32)
    k0 = min(B_local, n_gen)
    _, top_idx0 = _topk_smallest(values0, k0)
    chosen0 = neighbors0[top_idx0]
    if k0 < B_local:
        pad = jnp.broadcast_to(chosen0[-1:], (B_local - k0, state_size))
        states0 = jnp.concatenate([chosen0, pad], axis=0)
    else:
        states0 = chosen0[:B_local]

    states_global = jnp.broadcast_to(states0[None, :, :], (world_size, B_local, state_size))

    # Seed-move array (populates memmap[0] -- parent_local=0, parent_rank=0).
    top_idx0_i8 = top_idx0.astype(jnp.int8)
    if k0 < B_local:
        last_move = top_idx0_i8[k0 - 1]
        pad_moves = jnp.broadcast_to(last_move, (B_local - k0,))
        move0_full = jnp.concatenate([top_idx0_i8, pad_moves])
    else:
        move0_full = top_idx0_i8[:B_local]
    move0_full_np = np.asarray(move0_full)  # (B_local,) int8

    min_v_log = jnp.full((world_size, num_steps), 1e6, dtype=jnp.float32)
    found_step = jnp.full((world_size,), -1, dtype=jnp.int32)
    found_pos_local = jnp.full((world_size,), -1, dtype=jnp.int32)
    found_pos_rank = jnp.full((world_size,), -1, dtype=jnp.int32)
    verify_state = jnp.zeros((world_size, state_size), dtype=jnp.int8)

    # Early V0 check on the seed beam.
    eq0 = jnp.all(states0 == V0, axis=1)
    any0 = jnp.any(eq0)
    pos0 = jnp.argmax(eq0.astype(jnp.int32)).astype(jnp.int32)
    if bool(any0):
        pos0_h = int(pos0)
        if pos0_h < k0:
            seed_move = int(top_idx0[pos0_h])
        else:
            seed_move = int(top_idx0[k0 - 1])
        return {"found": True, "path_len": 1, "path_idx": [seed_move],
                "found_step": 0, "wall_s": 0.0}

    V0_hash_host = int(np.sum(np.asarray(V0).astype(np.int64) * np.asarray(hash_vec)))

    if parent_chunk is None:
        step_body = _build_step_body_v_only_packed(
            v_params, all_moves, V0, hash_vec, jnp.int64(V0_hash_host),
            B_local, world_size, K_per_peer, n_gen, state_size,
            dtype, int(internal_bs),
            pack_v_score=bool(pack_v_score),
            owner_hash_vec=owner_hash_vec,
        )
    else:
        step_body = _build_step_body_v_only_packed_streaming(
            v_params, all_moves, V0, hash_vec, jnp.int64(V0_hash_host),
            B_local, world_size, K_per_peer, n_gen, state_size,
            dtype, int(internal_bs),
            parent_chunk=int(parent_chunk),
            pack_v_score=bool(pack_v_score),
            owner_hash_vec=owner_hash_vec,
        )

    @partial(jax.jit, donate_argnums=(0, 1, 2, 3, 4, 5))  # NOT j_arr
    def step_fn(states, mv_log, fs, fpl, fpr, vstate, j_arr):
        try:
            from jax.experimental.shard_map import shard_map
        except ImportError:
            from jax import shard_map
        return shard_map(
            step_body,
            mesh=mesh,
            in_specs=(P("cores"), P("cores"), P("cores"), P("cores"),
                      P("cores"), P("cores"), P()),
            out_specs=(P("cores"),) * 7,  # states, packed, mv_log, fs, fpl, fpr, vstate
        )(states, mv_log, fs, fpl, fpr, vstate, j_arr)

    from jax.sharding import NamedSharding
    sharding_states = NamedSharding(mesh, P("cores"))

    states_d = jax.device_put(states_global, sharding_states)
    mv_log_d = jax.device_put(min_v_log, sharding_states)
    fs_d = jax.device_put(found_step, sharding_states)
    fpl_d = jax.device_put(found_pos_local, sharding_states)
    fpr_d = jax.device_put(found_pos_rank, sharding_states)
    vstate_d = jax.device_put(verify_state, sharding_states)

    # Host memmap for the packed tree.
    if tree_path is None:
        work_dir = "/kaggle/working"
        if not os.path.isdir(work_dir):
            work_dir = tempfile.gettempdir()
        fd, tree_path = tempfile.mkstemp(prefix="tree_", suffix=".u32", dir=work_dir)
        os.close(fd)
    tree_mm = np.memmap(
        tree_path, mode="w+", dtype=np.uint32,
        shape=(num_steps, world_size, B_local),
    )
    # Seed move at j=0: parent_local=0, parent_rank=0, move=top_idx0[k] (replicated).
    seed_packed = (move0_full_np.astype(np.uint32) << 26)  # (B_local,) uint32
    tree_mm[0, :, :] = np.broadcast_to(seed_packed[None, :], (world_size, B_local))

    import time

    # Pre-compile step_fn explicitly so compile time is separately measurable.
    if progress_every:
        print("[lower] tracing step_fn...", flush=True)
    t_lower_start = time.time()
    lowered = step_fn.lower(
        states_d, mv_log_d, fs_d, fpl_d, fpr_d, vstate_d, jnp.int32(1),
    )
    t_lower = time.time() - t_lower_start
    if progress_every:
        print(f"[lower] done in {t_lower:.1f}s", flush=True)
        print("[compile] compiling step_fn...", flush=True)
    t_compile_start = time.time()
    compiled_step = lowered.compile()
    t_compile = time.time() - t_compile_start
    if progress_every:
        print(f"[compile] done in {t_compile:.1f}s", flush=True)

    t_start = time.time()
    first_iter_t = None
    last_completed_step = 0
    fs_per_rank = np.asarray(fs_d)  # initial state (all -1), re-read in loop.

    try:
        for j in range(1, num_steps):
            t_iter = time.time()
            j_arr = jnp.int32(j)
            (states_d, packed_d, mv_log_d,
             fs_d, fpl_d, fpr_d, vstate_d) = compiled_step(
                states_d, mv_log_d, fs_d, fpl_d, fpr_d, vstate_d, j_arr,
            )
            # Block on device computation (any output works; fs is smallest).
            jax.block_until_ready(fs_d)
            t_device = time.time() - t_iter

            # Sync + transfer packed backptr to host memmap.
            t_copy_start = time.time()
            packed_h = np.asarray(packed_d)
            t_copy = time.time() - t_copy_start

            t_write_start = time.time()
            tree_mm[j, :, :] = packed_h
            t_write = time.time() - t_write_start

            fs_per_rank = np.asarray(fs_d)
            last_completed_step = j

            if first_iter_t is None:
                first_iter_t = time.time() - t_iter

            if progress_every and (j == 1 or j % progress_every == 0 or np.any(fs_per_rank >= 0)):
                print(
                    f"[step {j:03d}/{num_steps-1}] device={t_device:.1f}s "
                    f"copy={t_copy:.2f}s write={t_write:.2f}s "
                    f"total={time.time()-t_iter:.1f}s "
                    f"fs={fs_per_rank.tolist()}",
                    flush=True,
                )

            if np.any(fs_per_rank >= 0):
                break

        tree_mm.flush()

        fpl_per_rank = np.asarray(fpl_d)
        mv_per_rank = np.asarray(mv_log_d)

        INT_MAX = 2 ** 30
        fs_signed = np.where(fs_per_rank >= 0, fs_per_rank, INT_MAX)
        global_min_step = int(fs_signed.min())
        if global_min_step >= INT_MAX:
            return {"found": False, "path_len": 0, "path_idx": [],
                    "found_step": -1, "wall_s": time.time() - t_start,
                    "first_iter_s": first_iter_t,
                    "last_completed_step": last_completed_step,
                    "lower_s": t_lower, "compile_s": t_compile,
                    "min_v_trajectory_rank0": mv_per_rank[0].tolist()}

        winner_ranks = np.where(fs_signed == global_min_step)[0]
        winner_rank = int(winner_ranks[0])
        fs_h = int(fs_per_rank[winner_rank])
        fpl_h = int(fpl_per_rank[winner_rank])

        # Walkback through the packed memmap.
        PL_MASK = (1 << 23) - 1
        path_idx = []
        cur_rank = winner_rank
        cur_pos = fpl_h
        for jj in range(fs_h, -1, -1):
            rec = int(tree_mm[jj, cur_rank, cur_pos])
            parent_local = rec & PL_MASK
            parent_rank = (rec >> 23) & 0x7
            move = (rec >> 26) & 0x1F
            path_idx.append(int(move))
            if jj > 0:
                cur_rank = int(parent_rank)
                cur_pos = int(parent_local)
        path_idx.reverse()

        return {
            "found": True,
            "path_len": len(path_idx),
            "path_idx": path_idx,
            "found_step": fs_h,
            "found_pos_local": fpl_h,
            "found_pos_rank": winner_rank,
            "wall_s": time.time() - t_start,
            "first_iter_s": first_iter_t,
            "last_completed_step": last_completed_step,
            "lower_s": t_lower, "compile_s": t_compile,
            "min_v_trajectory_rank0": mv_per_rank[0].tolist(),
        }
    finally:
        # Always release memmap + delete the tree file.
        try:
            del tree_mm
        except Exception:
            pass
        gc.collect()
        try:
            if os.path.exists(tree_path):
                os.unlink(tree_path)
        except OSError:
            pass


In [ ]:
# Cell 8: Sym ensemble setup. Deterministic K_SYM rotations from SYM_SEED=0,
# drawn from the 720-element symmetry group, then sliced by SYM_POSITIONS.
SYM_SEED = 0

rotations_path = rotations_dataset_root / "rotations_720.npy"
rotations_np = np.load(rotations_path)
assert rotations_np.shape == (720, 120), \
    f"expected rotations_720.npy shape (720, 120); got {rotations_np.shape}"
print(f"rotations: {rotations_np.shape} dtype={rotations_np.dtype}")

identity_arr = np.arange(STATE_SIZE, dtype=rotations_np.dtype)
identity_idx = None
for i in range(rotations_np.shape[0]):
    if np.array_equal(rotations_np[i], identity_arr):
        identity_idx = i
        break
assert identity_idx is not None

# Deterministic full K_SYM-rotation list. Identity at position 0, then
# K_SYM-1 distinct non-identity rotations drawn with seed=0. Both shards
# of a split run hit this same code path with the same K_SYM, so they
# agree on what every position 0..K_SYM-1 means.
rng_sym = np.random.default_rng(SYM_SEED)
other_idxs = [i for i in range(rotations_np.shape[0]) if i != identity_idx]
auto_chosen = [identity_idx]
if K_SYM > 1:
    auto_chosen += list(rng_sym.choice(other_idxs, size=K_SYM - 1, replace=False).tolist())
print(f"K_SYM={K_SYM} SYM_SEED={SYM_SEED} full auto-selection: {auto_chosen}")

# This kernel runs only the slice asked for in Cell 1.
chosen_rot_idxs = [auto_chosen[p] for p in SYM_POSITIONS]
print(f"this kernel sym positions: {SYM_POSITIONS}")
print(f"this kernel rotation idxs: {chosen_rot_idxs}")

ROTATIONS = []
move_name_to_idx = {n: i for i, n in enumerate(MOVE_NAMES)}
perm_to_name = {tuple(v): n for n, v in GENERATORS.items()}

for sym_pos, rot_idx in zip(SYM_POSITIONS, chosen_rot_idxs):
    R = tuple(int(x) for x in rotations_np[rot_idx])
    R_inv = tuple(int(x) for x in np.argsort(rotations_np[rot_idx]))
    conj_idx = [-1] * N_GEN
    for nm in MOVE_NAMES:
        g = GENERATORS[nm]
        conj_perm = tuple(R_inv[g[R[i]]] for i in range(STATE_SIZE))
        if conj_perm not in perm_to_name:
            raise ValueError(f"rotation {rot_idx} not a symmetry")
        conj_idx[move_name_to_idx[nm]] = move_name_to_idx[perm_to_name[conj_perm]]
    ROTATIONS.append({
        "sym_pos": sym_pos,           # position in the full K_SYM ensemble
        "rot_idx": rot_idx,           # absolute idx into rotations_720.npy
        "R": R, "R_inv": R_inv, "conj_idx": conj_idx,
    })

def apply_rotation_state(state, R, R_inv):
    return tuple(R[state[R_inv[i]]] for i in range(len(state)))

PID_ROT_PAIRS = []
for pid in TEST_PIDS:
    for ri in range(len(ROTATIONS)):
        PID_ROT_PAIRS.append((pid, ri))
print(f"total (pid, sym_pos) pairs to solve: {len(PID_ROT_PAIRS)}")


In [ ]:
# Cell 9: Load V checkpoint, build JAX params dict, set up mesh.
print("loading AZ v4 V (V teacher, V-only) ...")
v_params = load_params_from_pt(v_path, hidden_dims=(2048, 512))
print(f"  V params: {num_params(v_params):,}")

hash_vec_np = make_hash_vec(STATE_SIZE, seed=0)
hash_vec = jnp.asarray(hash_vec_np)

# Cheaper uint32 hash used only for owner routing (not for dedup / V0).
# Owner partition needs uniform 0..world_size-1, not collision resistance.
_owner_rng = np.random.default_rng(12345)
owner_hash_vec_np = _owner_rng.integers(
    0, np.iinfo(np.uint32).max, size=STATE_SIZE, dtype=np.uint32,
)
owner_hash_vec = jnp.asarray(owner_hash_vec_np)

mesh = make_mesh(devices)
print(f"mesh: {mesh}")


In [ ]:
# Cell 10: SPMD V-only solve loop — 1 pid at a time, all 8 cores cooperating.
# Shard-aware output filenames so two collaborators can download artifacts
# without colliding.
SLICE_TAG = f"residuals_b1m_k{K_SYM}_sym{min(SYM_POSITIONS)}_{max(SYM_POSITIONS)}"
if use_niss:
    SLICE_TAG += "_niss"
PARTIAL_PATH = f"/kaggle/working/share_jax_partial_{SLICE_TAG}.json"
FINAL_PATH   = f"/kaggle/working/share_jax_final_{SLICE_TAG}.json"
print(f"SLICE_TAG: {SLICE_TAG}")
print(f"partial: {PARTIAL_PATH}")
print(f"final:   {FINAL_PATH}")

def _verify(initial_state, path_idx_orig):
    cur = list(initial_state)
    for m in path_idx_orig:
        gen = GENERATORS[MOVE_NAMES[m]]
        cur = [cur[g] for g in gen]
    return tuple(cur) == SOLVED

results_list = []
t_pipeline = time.time()
for i, (pid, rot_i) in enumerate(PID_ROT_PAIRS):
    rot = ROTATIONS[rot_i]
    sym_pos = rot["sym_pos"]
    rot_idx_abs = rot["rot_idx"]
    s0 = PID_STATES[pid]
    s_to_solve = apply_rotation_state(s0, rot["R"], rot["R_inv"])

    t_iter = time.time()

    # Phase B: per-pid memmap tree -- separate path per direction so fwd and inv
    # don't stomp each other. Tagged by sym_pos rather than local rot_i so
    # logs are unambiguous across forks.
    tree_path_fwd = f"/kaggle/working/tree_pid{pid}_sym{sym_pos}_fwd.u32"
    tree_path_inv = f"/kaggle/working/tree_pid{pid}_sym{sym_pos}_inv.u32"

    # --- Forward direction: solve s_to_solve directly ---
    t0 = time.time()
    try:
        r_fwd = beam_solve_v_only_spmd_packed(
            list(s_to_solve), v_params,
            all_moves, V0, hash_vec, mesh,
            B_local=B_LOCAL, K_per_peer=K_PER_PEER,
            n_gen=N_GEN, state_size=STATE_SIZE,
            num_steps=NUM_STEPS, dtype=jnp.bfloat16,
            internal_bs=INTERNAL_BS,
            tree_path=tree_path_fwd,
            parent_chunk=PARENT_CHUNK,
            pack_v_score=PACK_V_SCORE,
            progress_every=PROGRESS_EVERY,
            owner_hash_vec=owner_hash_vec,
        )
    except Exception as e:
        import traceback
        print(f"[{i+1}/{len(PID_ROT_PAIRS)}] pid={pid} sym_pos={sym_pos} FWD EXCEPTION: {e}")
        traceback.print_exc()
        results_list.append({
            "pid": pid, "sym_pos": sym_pos, "rot_idx": rot_idx_abs,
            "found": False, "verify_ok": False,
            "path_len": 0, "path_idx_orig": [], "wall_s": time.time() - t0,
            "direction": "fwd", "error": str(e),
        })
        for _tp in (tree_path_fwd, tree_path_inv):
            if os.path.exists(_tp):
                try: os.unlink(_tp)
                except OSError: pass
        continue
    wall_fwd = time.time() - t0

    # --- NISS: also solve invert_state(s_to_solve), invert the path back ---
    r_inv = None
    inv_path_rotated = None
    wall_inv = 0.0
    if use_niss:
        s_inv = np.argsort(np.asarray(s_to_solve, dtype=np.int32)).tolist()
        t1 = time.time()
        try:
            r_inv = beam_solve_v_only_spmd_packed(
                s_inv, v_params,
                all_moves, V0, hash_vec, mesh,
                B_local=B_LOCAL, K_per_peer=K_PER_PEER,
                n_gen=N_GEN, state_size=STATE_SIZE,
                num_steps=NUM_STEPS, dtype=jnp.bfloat16,
                internal_bs=INTERNAL_BS,
                tree_path=tree_path_inv,
                parent_chunk=PARENT_CHUNK,
                pack_v_score=PACK_V_SCORE,
                progress_every=PROGRESS_EVERY,
                owner_hash_vec=owner_hash_vec,
            )
        except Exception as e:
            import traceback
            print(f"[{i+1}/{len(PID_ROT_PAIRS)}] pid={pid} sym_pos={sym_pos} INV EXCEPTION (continuing with fwd only): {e}")
            traceback.print_exc()
            r_inv = {"found": False, "path_idx": [], "found_step": -1}
        wall_inv = time.time() - t1
        if r_inv.get("found"):
            # Path solves s_inv. Invert it (reverse + swap each move with its inverse)
            # to get a path that solves s_to_solve (still in the rotated frame).
            inv_path_rotated = [int(INV_MOVE_IDX[m]) for m in reversed(list(r_inv["path_idx"]))]

    # --- Build candidate list, un-rotate, and verify each before picking min ---
    # Verifying both means a buggy inversion can never drop a valid fwd path:
    # we keep the shorter VERIFIED candidate, falling back to fwd if inv breaks.
    conj_idx = rot["conj_idx"]
    raw_candidates = []
    if r_fwd.get("found"):
        raw_candidates.append(("fwd", list(r_fwd["path_idx"]), r_fwd.get("found_step", -1)))
    if inv_path_rotated is not None:
        raw_candidates.append(("inv", inv_path_rotated, r_inv.get("found_step", -1)))

    verified_candidates = []
    for tag, rot_path, fstep in raw_candidates:
        orig = [conj_idx[m] for m in rot_path]
        if _verify(s0, orig):
            verified_candidates.append((tag, rot_path, orig, fstep))
        else:
            print(f"  WARN pid={pid} sym_pos={sym_pos} {tag} path failed verify ({len(orig)} moves) - dropping")

    if verified_candidates:
        direction, best_rot_path, path_orig, found_step = min(
            verified_candidates, key=lambda x: len(x[1])
        )
        found_any = True
        verify_ok = True
    else:
        direction, best_rot_path, path_orig, found_step = "none", [], None, -1
        found_any = False
        verify_ok = False

    # Clean up memmap files for this iteration.
    for _tp in (tree_path_fwd, tree_path_inv):
        if os.path.exists(_tp):
            try: os.unlink(_tp)
            except OSError: pass

    wall = time.time() - t_iter
    rec = {
        "pid": pid,
        "sym_pos": sym_pos,           # logical position in the full K_SYM ensemble
        "rot_idx": rot_idx_abs,       # absolute idx into rotations_720.npy
        "found": found_any, "verify_ok": verify_ok,
        "path_len": len(path_orig) if path_orig is not None else 0,
        "path_idx_orig": path_orig if verify_ok else [],
        "path_idx_rotated": best_rot_path,
        "direction": direction,
        "wall_s": wall,
        "wall_fwd_s": wall_fwd,
        "wall_inv_s": wall_inv,
        "found_step": found_step,
        "fwd_found": bool(r_fwd.get("found")),
        "fwd_path_len": len(r_fwd["path_idx"]) if r_fwd.get("found") else 0,
        "inv_found": bool(r_inv.get("found")) if r_inv is not None else None,
        "inv_path_len": len(inv_path_rotated) if inv_path_rotated is not None else 0,
        "first_iter_s": r_fwd.get("first_iter_s"),
        "last_completed_step": r_fwd.get("last_completed_step"),
    }
    results_list.append(rec)
    _inv_str = f" inv={rec['inv_path_len']:>3}" if use_niss else ""
    print(f"[{i+1}/{len(PID_ROT_PAIRS)}] pid={pid:>4} sym_pos={sym_pos} rot_idx={rot_idx_abs} "
          f"found={found_any} verify={verify_ok} "
          f"fwd={rec['fwd_path_len']:>3}{_inv_str} -> {direction} {rec['path_len']:>3} "
          f"wall={wall:.1f}s (fwd={wall_fwd:.1f}s inv={wall_inv:.1f}s)",
          flush=True)

    with open(PARTIAL_PATH, "w") as f:
        json.dump(results_list, f)

with open(FINAL_PATH, "w") as f:
    json.dump(results_list, f, indent=2)
print(f"\n== pipeline done in {time.time() - t_pipeline:.1f}s, {len(results_list)} pairs ==")


In [ ]:
# Cell 11: Per-pid min over THIS kernel's slice of positions + submission CSV.
# Owner min-merges this CSV against the other shard's CSV (and current best)
# locally with megaminx/scripts/16_merge_rescue.py.
import csv as _csv

by_pid = {}
for r in results_list:
    by_pid.setdefault(r["pid"], []).append(r)

print()
print(f"== per-pid (min over this kernel's sym positions {SYM_POSITIONS}) ==")
print(f"{'pid':>6}  {'best_pos':>8}  {'best_rot':>8}  {'best_path':>9}  {'found/N':>10}  {'wall':>9}")
final_per_pid = {}
total_path = 0
n_solved = 0
for pid in sorted(by_pid.keys()):
    rs = by_pid[pid]
    valid = [r for r in rs if r["verify_ok"]]
    if valid:
        best = min(valid, key=lambda r: r["path_len"])
        final_per_pid[pid] = {"found": True, "best_path_len": best["path_len"],
                              "best_path_idx": best["path_idx_orig"],
                              "best_sym_pos": best["sym_pos"],
                              "best_rot_idx": best["rot_idx"]}
        total_path += best["path_len"]
        n_solved += 1
        wall_max = max(r["wall_s"] for r in rs)
        print(f"{pid:>6}  {best['sym_pos']:>8}  {best['rot_idx']:>8}  {best['path_len']:>9}  "
              f"{len(valid):>4}/{len(rs):<4}  {wall_max:>9.1f}")
    else:
        final_per_pid[pid] = {"found": False}
        wall_max = max(r["wall_s"] for r in rs)
        print(f"{pid:>6}  {'-':>8}  {'-':>8}  {'-':>9}  {'0':>4}/{len(rs):<4}  {wall_max:>9.1f}")

print()
print(f"solved (any position in this slice): {n_solved}/{len(by_pid)}")
print(f"total_path (best per pid, this slice only): {total_path}")

csv_path = f"/kaggle/working/share_jax_submission_{SLICE_TAG}.csv"
with open(csv_path, "w", newline="") as f:
    w = _csv.writer(f)
    w.writerow(["initial_state_id", "path"])
    for pid in sorted(by_pid.keys()):
        info = final_per_pid[pid]
        if info["found"]:
            names = [MOVE_NAMES[m] for m in info["best_path_idx"]]
            w.writerow([pid, ".".join(names)])
        else:
            w.writerow([pid, ""])
print(f"wrote {csv_path}")
print()
print(f"== Send share_jax_submission_{SLICE_TAG}.csv (and share_jax_final_{SLICE_TAG}.json) ==")
print(f"== back to the kernel owner for cross-shard min-merge.                            ==")
